# Asset-generation meta prompt (Step 1b)

Sends the meta prompt (`assetgen_meta/meta_prompt_v1.txt`) to the meta model several times.
Each reply holds a **methodology** and a ready **execution prompt** for the executor.

- Outputs: `assetgen_meta/runs/meta_<prompt sha>_<model>/sample_<i>/` holding `raw.txt`, `methodology.txt`, `exec_prompt.txt`, `meta.json` and `checks.json`.
- A finished sample is never overwritten. Re-running skips it, so an interrupted run resumes.
- A changed meta prompt gets a new sha, and so a new folder.
- The executor schema is nested. `meta_tools.flatten()` turns it into the format `eval_assets` already scores, so the scorer stays unchanged.
- Nothing here calls the executor. Screening the execution prompts (gpt-5-mini, 1 run each on the 15 modules) is the next step.

In [34]:
# Setup
import sys, json, time
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

print("loaded:", load_dotenv("API.env"))
sys.path.insert(0, "assetgen_meta")
import importlib, meta_tools as mt
importlib.reload(mt)

META_MODEL = "gpt-6-astra"    # writes the methods; the executor stays gpt-5-mini
EFFORT     = "high"
N_SAMPLES  = 1                # seeded revision: one revision per arm (the from-scratch screens used 3)
SCREEN_REPEATS = 3            # seeded arms get 3 executor runs directly: no pick, so no winner's curse
MAX_TOKENS = 65536
USER_MSG   = "Write the two parts now, following every [FIXED] section."

mt.selftest()                 # adapter, scorer, reply split and prompt checks, no API calls
client = OpenAI()

loaded: True
selftest PASS: adapter, scorer (TP 2 / FP 1 on wdt), reply split, 2031 corpus names, negative prompt caught 4 problems


In [57]:
# Load and check the meta prompt; stop here if a check fails
META_PATH = Path("assetgen_meta/meta_prompt_v5s.txt")   # seeded revision of m7e194es0: X1, X3, X4 rescoped, X5 (see Step 1b log)
META = META_PATH.read_text(encoding="utf-8")
SEED_PATH = Path("assetgen_meta/runs/meta_7e194e6254be_gpt-6-astra/sample_0/exec_prompt.txt")
SEED = SEED_PATH.read_text(encoding="utf-8") if "<<SEED_EXECUTION_PROMPT>>" in META else None
if SEED:
    META = META.replace("<<SEED_EXECUTION_PROMPT>>", SEED)   # the sha below covers instructions + seed
NAMES = mt.corpus_names()
chk = mt.check_prompt(META, NAMES, META_PATH.name)
print(f"{META_PATH.name}: {len(META)} chars, sha {mt.sha12(META)}")
print("checks:", "PASS" if chk["ok"] else chk["problems"])
assert chk["ok"], "fix the meta prompt before calling the model"
OUT = Path("assetgen_meta/runs") / f"meta_{mt.sha12(META)}_{META_MODEL}"
print("output folder:", OUT)

meta_prompt_v5s.txt: 33054 chars, sha 43f96512e1e7
checks: PASS
output folder: assetgen_meta\runs\meta_43f96512e1e7_gpt-6-astra


In [58]:
# Confirm the model name exists before spending anything
try:
    print("model found:", client.models.retrieve(META_MODEL).id)
except Exception as e:
    print(f"'{META_MODEL}' not found ({type(e).__name__}). gpt-5 family models on this key:")
    print(sorted(m.id for m in client.models.list() if m.id.startswith("gpt-5")))
    raise

model found: gpt-6-astra


In [59]:
# Generate N_SAMPLES methods + execution prompts
OUT.mkdir(parents=True, exist_ok=True)
for i in range(N_SAMPLES):
    d = OUT / f"sample_{i}"
    if (d / "meta.json").exists():
        print(f"sample_{i}: exists, skipped"); continue
    d.mkdir(exist_ok=True)
    t0 = time.time()
    text, usage, status = mt.call_text(client, META_MODEL, META, USER_MSG, EFFORT, MAX_TOKENS)
    (d / "raw.txt").write_text(text, encoding="utf-8")
    rec = {"meta_prompt": META_PATH.name, "meta_prompt_sha": mt.sha12(META), "model": META_MODEL,
           "effort": EFFORT, "status": status, "usage": usage, "seconds": round(time.time() - t0),
           "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
    try:
        meth, exe = mt.split_reply(text)
        (d / "methodology.txt").write_text(meth, encoding="utf-8")
        (d / "exec_prompt.txt").write_text(exe, encoding="utf-8")
        rec["exec_prompt_sha"] = mt.sha12(exe)
        chk = mt.check_exec_prompt(exe, NAMES, f"sample_{i}")
    except ValueError as e:
        chk = {"label": f"sample_{i}", "ok": False, "problems": [f"reply split: {e}"]}
    (d / "checks.json").write_text(json.dumps(chk, indent=1), encoding="utf-8")
    (d / "meta.json").write_text(json.dumps(rec, indent=1), encoding="utf-8")
    print(f"sample_{i}: status {status}, {usage['out']} out tokens ({usage['reasoning']} reasoning), "
          f"{rec['seconds']} s, checks {'PASS' if chk['ok'] else 'FAIL'}")

sample_0: status completed, 14238 out tokens (5498 reasoning), 186 s, checks PASS


In [60]:
# Seed preservation (evaluation protocol): how much of the seed survived verbatim, and what changed.
# Read each sample's change log in methodology.txt as well; anything changed outside the listed changes breaks the seeded design.
if SEED:
    for d in sorted(OUT.glob("sample_*")):
        exe = d / "exec_prompt.txt"
        if exe.exists():
            r = mt.seed_diff(SEED, exe.read_text(encoding="utf-8"))
            print(f"{d.name}: {r['kept']}/{r['seed_sentences']} seed sentences kept verbatim ({r['kept_share']:.0%}); "
                  f"{r['removed_n']} removed or edited, {r['added_n']} added")
            for s in r["removed"][:15]:
                print("   - removed/edited:", s[:170])
else:
    print("not a seeded meta prompt")

sample_0: 272/292 seed sentences kept verbatim (93%); 20 removed or edited, 66 added
   - removed/edited: An element that only passes the value along, or gates it on the way, carries the asset but is not its structural reference.
   - removed/edited: An input port whose value this entity itself reads, to store it, compute from it, or decide on it, is where that value enters the entity:
   - removed/edited: it is a structural reference of the concept it carries, with realization "sets".
   - removed/edited: Clock, clock-enable and reset inputs, and ports whose record type carries complete bus transactions, are not structural references in this way.
   - removed/edited: Elements that only pass the value along or gate it between those points are not primary, but secondary assets.
   - removed/edited: An input port whose value this entity itself reads, to store it, compute from it, or decide on it, is where that value is set.
   - removed/edited: an element that only passes a primary asset

In [61]:
# Summary: which samples are usable for screening
for d in sorted(OUT.glob("sample_*")):
    if not (d / "meta.json").exists():
        continue
    rec = json.loads((d / "meta.json").read_text(encoding="utf-8"))
    chk = json.loads((d / "checks.json").read_text(encoding="utf-8"))
    exe = d / "exec_prompt.txt"
    size = len(exe.read_text(encoding="utf-8")) if exe.exists() else 0
    print(f"{d.name}: {'USABLE' if chk['ok'] else 'NOT USABLE'}  exec prompt {size} chars  "
          f"sha {rec.get('exec_prompt_sha', '-')}  status {rec['status']}")
    for p in chk["problems"]:
        print("    -", p)

sample_0: USABLE  exec prompt 32419 chars  sha 25dcbe668d30  status completed


## Screening on the executor (gpt-5-mini)

The executor is set up exactly like `v2x3r8`: gpt-5-mini, effort high, JSON mode, a 65,536-token ceiling, **comment-stripped RTL only**, the same user-message wrapper, and **no validation retry** (the baseline had none either). Validation problems are recorded in each run's `_validation.json`, not acted on.

1. **Screen:** 1 run per usable sample goes to `assets_tuning18_m<meta sha>s<i>_r0/`. Each module is written three ways: `<module>.json` flattened for the scorer, `_nested/` holding your schema, and `_raw/` holding the reply verbatim.
2. **Score:** `eval_assets`, strict matching, on the modules shared with the ground truth. The same code and denominator as the baseline.
3. **Pick:** the highest precision with recall ≥ 0.83.
4. **Confirm:** 3 fresh runs of the winner, as version `<winner>c`. Guarded by `RUN_CONFIRM`.
5. **Extras:** the majority vote across the 3 runs, reported as a separate number; 40 random false positives exported to CSV for grouping by hand.

Each run makes 18 calls. The baseline's run folders were written about 9 minutes apart.

In [62]:
# Screening setup: one version per usable sample

importlib.reload(mt)
EXEC_MODEL   = "gpt-5-mini"
BASELINE     = "v2x3r8"          # precision 0.296, recall 0.853: mean of 3 runs, 15 modules, 111 ground-truth entries
RECALL_FLOOR = 0.83
MODULES      = mt.rtl_modules()   # the 18 tuning modules in RTL_data/
mt.recheck(OUT, NAMES)           # re-run the prompt checks with the current meta_tools
META_SHA = mt.sha12(META)
CANDIDATES = {}
for d in sorted(OUT.glob("sample_*")):
    chk = json.loads((d / "checks.json").read_text(encoding="utf-8")) if (d / "checks.json").exists() else {"ok": False}
    if chk["ok"] and (d / "exec_prompt.txt").exists():
        CANDIDATES[f"m{META_SHA[:6]}s{d.name.split('_')[1]}"] = d / "exec_prompt.txt"
print(f"{len(MODULES)} modules; {len(CANDIDATES)} usable candidates:")
for v, p in CANDIDATES.items():
    print(f"  {v}  <-  {p}")

sample_0: PASS
18 modules; 1 usable candidates:
  m43f965s0  <-  assetgen_meta\runs\meta_43f96512e1e7_gpt-6-astra\sample_0\exec_prompt.txt


In [63]:
# Screening: SCREEN_REPEATS runs per candidate (repeats 0..n-1). Cached modules are skipped, so re-running resumes.
for v, p in CANDIDATES.items():
    for rep in range(SCREEN_REPEATS):
        t0 = time.time()
        s = mt.run_version(client, v, rep, p.read_text(encoding="utf-8"), MODULES, model=EXEC_MODEL,
                           extra_meta={"meta_sample": str(p.parent), "meta_model": META_MODEL, "stage": "screen"})
        print(f"{v} r{rep}: {s['modules_done']}/{len(MODULES)} modules, {round(time.time() - t0)} s, tokens {s['tokens']}, "
              f"validation issues {s['issues']} {s['issue_kinds']}")
        for k, n in s["notes"].items():
            if n not in ("ok", "cached"):
                print(f"    {k}: {n}")

m43f965s0 r0: 18/18 modules, 770 s, tokens {'in': 174749, 'out': 340762}, validation issues 1 {'ungrounded': 1}
m43f965s0 r1: 18/18 modules, 663 s, tokens {'in': 174749, 'out': 336401}, validation issues 1 {'ungrounded': 1}
m43f965s0 r2: 18/18 modules, 1097 s, tokens {'in': 174749, 'out': 335036}, validation issues 2 {'ungrounded': 2}


In [64]:
# Score the screen against the baseline, and pick the winner
done = [v for v in CANDIDATES if all((mt.run_dir(v, r) / f"{s}.json").exists()
                                     for r in range(SCREEN_REPEATS) for s, _ in MODULES)]
if set(CANDIDATES) - set(done):
    print("NOT scored (incomplete run; re-run the screening cell):", sorted(set(CANDIDATES) - set(done)))
PREVIOUS = ["m4bc606s0", "m4bc606s1", "m4bc606s2",    # screen 1 (meta v1)
            "m7e194es0", "m7e194es1", "m7e194es2",    # screen 2 (meta v2)
            "med96f4s0", "med96f4s1", "med96f4s2",    # screen 3 (meta v3)
            "m7e194es0c",                             # confirmation of screen-2 s0 (3 runs; the seed)
            "m2a78a4s0", "m55eea9s0"]                # seeded arms A and B (meta v4s, 3 runs each)
PREVIOUS = [v for v in PREVIOUS if v not in done and mt.run_dir(v, 0).exists()]
common, rows, n_gt = mt.score_versions(done + PREVIOUS, BASELINE)
print(f"scored on {len(common)} modules common to all runs and the ground truth; {n_gt} ground-truth entries (exact)\n")
print(f"{'version':14s} {'runs':>4s} {'emitted':>7s} {'precision':>9s} {'recall':>7s} {'F1':>6s}")
for v, r in sorted(rows.items(), key=lambda kv: -kv[1]["P"]):
    print(f"{v:14s} {r['runs']:4d} {r['emit']:7.1f} {r['P']:9.3f} {r['R']:7.3f} {r['F1']:6.3f}"
          + ("   <- baseline (mean of its runs)" if v == BASELINE else "   (earlier screen)" if v in PREVIOUS else ""))
for v in done:
    print(f"{v} realization counts: {mt.realization_counts(v, 0)}")
ok = [v for v in done if rows[v]["R"] >= RECALL_FLOOR]
WINNER = max(ok, key=lambda v: rows[v]["P"]) if ok else None
print(f"\nWINNER (highest precision with recall >= {RECALL_FLOOR}): {WINNER}")
if WINNER:
    print(f"precision {rows[WINNER]['P'] - rows[BASELINE]['P']:+.3f} vs {BASELINE} (one screening run vs the baseline's mean)")

# F13: a CODE post-filter (closed-set names only, one row per element), applied identically to every version.
# A separately labelled number; the pick above uses the unfiltered scores.
_c13, rows13, _n13 = mt.score_versions(done + PREVIOUS, BASELINE, post=mt.f13_filter)
print("\nWITH F13 CODE FILTER (separate number, not the pick basis):")
for v, r in sorted(rows13.items(), key=lambda kv: -kv[1]["P"]):
    print(f"{v:14s} precision {r['P']:.3f}  recall {r['R']:.3f}  (unfiltered {rows[v]['P']:.3f} / {rows[v]['R']:.3f})"
          + ("   <- baseline" if v == BASELINE else "   (earlier screen)" if v in PREVIOUS else ""))

# ---- Evaluation-protocol additions (2026-09-19). All are separate numbers; the decision uses the unfiltered scores. ----
_cc, rowsC, _nc = mt.score_versions(done + PREVIOUS, BASELINE, post=mt.convention_filter)
print("\nWITH CONVENTION FILTER (annotator conventions, not theory: transaction-record elements, clock/reset inputs):")
for v, r in sorted(rowsC.items(), key=lambda kv: -kv[1]["P"]):
    print(f"{v:14s} precision {r['P']:.3f}  recall {r['R']:.3f}")
REACHABLE = n_gt - 5   # absent from this RTL or never read (2026-09-19): cache inval_i, cache_o.cmd_dir, cfu active_i, cfu rs3_i, trng fifo.free
print(f"\nRecall against reachable ground truth ({REACHABLE} of {n_gt}; separate number):")
for v in done + ["m7e194es0c", BASELINE]:
    if v in rows:
        print(f"  {v:14s} {rows[v]['R'] * n_gt / REACHABLE:.3f}")
for v in done:
    rec, still = mt.recovered_from(v, "m7e194es0c")
    print(f"\n{v}: recovers {len(rec)} of the {len(rec) + len(still)} GT elements the seed (m7e194es0c) missed in most runs:")
    print("   recovered:", sorted(f"{m}:{e}" for m, e in rec))
    print("   still missed:", sorted(f"{m}:{e}" for m, e in still))
BEST_COUNTED = 0.296   # best counted Step 1b round so far: round 1, the baseline
for v in done:
    r = rows[v]
    if r["runs"] >= 3:
        verdict = "counts" if r["R"] >= RECALL_FLOOR else "does NOT count (recall below floor)"
        print(f"\nStep 1b: {v}, mean of {r['runs']} runs: precision {r['P']:.3f} (gain {r['P'] - BEST_COUNTED:+.3f} over "
              f"{BEST_COUNTED}), recall {r['R']:.3f} -> {verdict}")

scored on 15 modules common to all runs and the ground truth; 111 ground-truth entries (exact)

version        runs emitted precision  recall     F1
m7e194es2         1   214.0     0.383   0.739  0.505   (earlier screen)
med96f4s1         1   202.0     0.381   0.694  0.492   (earlier screen)
m7e194es0         1   242.0     0.380   0.829  0.521   (earlier screen)
m7e194es0c        3   230.3     0.379   0.787  0.512   (earlier screen)
med96f4s2         1   193.0     0.373   0.649  0.474   (earlier screen)
m7e194es1         1   211.0     0.365   0.694  0.478   (earlier screen)
med96f4s0         1   220.0     0.364   0.721  0.483   (earlier screen)
m43f965s0         3   264.3     0.339   0.808  0.478
v2x3r8            3   319.7     0.296   0.853  0.439   <- baseline (mean of its runs)
m2a78a4s0         3   288.3     0.284   0.739  0.410   (earlier screen)
m4bc606s2         1   165.0     0.273   0.405  0.326   (earlier screen)
m55eea9s0         3   316.0     0.266   0.757  0.394   (earlier 

## Confirmation: iteration 1

Read the screening table first, then set `RUN_CONFIRM = True`. The winner gets **3 fresh runs** under the version name `<winner>c`. Its screening run is not reused, because it was picked partly for being a good run.

In [65]:
RUN_CONFIRM = False               # set True after reading the screening table
OVERRIDE_S0 = False               # the 2026-09-19 override that confirmed m7e194es0 (done: runs m7e194es0c); kept for the record
if OVERRIDE_S0:
    WINNER = "m7e194es0"
    CANDIDATES[WINNER] = Path("assetgen_meta/runs/meta_7e194e6254be_gpt-6-astra/sample_0/exec_prompt.txt")
SCREENED_TOTAL = len(list(Path("assetgen_meta/runs").glob("*/sample_*/exec_prompt.txt")))   # every candidate generated so far
if RUN_CONFIRM:
    assert WINNER, "no candidate met the recall floor"
    CONFIRM = WINNER + "c"
    prompt = CANDIDATES[WINNER].read_text(encoding="utf-8")
    for rep in range(3):
        s = mt.run_version(client, CONFIRM, rep, prompt, MODULES, model=EXEC_MODEL,
                           extra_meta={"meta_sample": str(CANDIDATES[WINNER].parent), "meta_model": META_MODEL,
                                       "stage": "confirm", "screened_total": SCREENED_TOTAL,
                                       "override": "pick rule overridden: screening recall 92/111, one element under the floor"})
        print(f"{CONFIRM} r{rep}: {s['modules_done']}/{len(MODULES)} modules, validation issues {s['issues']} {s['issue_kinds']}")
        for k, n in s["notes"].items():
            if n not in ("ok", "cached"):
                print(f"    {k}: {n}")
else:
    print("RUN_CONFIRM is False: confirmation skipped")

RUN_CONFIRM is False: confirmation skipped


In [66]:
# Iteration 1 result: mean of the 3 confirmation runs; the majority vote is a separate number
import eval_assets as ea
CONFIRM = (WINNER + "c") if WINNER else None
if CONFIRM and len(list(Path(".").glob(f"assets_tuning18_{CONFIRM}_r*"))) >= 3:
    common, rows, n_gt = mt.score_versions([CONFIRM], BASELINE)
    b, c = rows[BASELINE], rows[CONFIRM]
    print(f"{len(common)} modules, {n_gt} ground-truth entries (exact)")
    print(f"{BASELINE:12s} precision {b['P']:.3f}  recall {b['R']:.3f}  (mean of {b['runs']} runs)")
    print(f"{CONFIRM:12s} precision {c['P']:.3f}  recall {c['R']:.3f}  (mean of {c['runs']} runs; per run {c['per_run']})")
    voted, _ = mt.majority(ea.collect([CONFIRM])[CONFIRM])
    sv = ea.score(voted, ea.load_refs()["gt"], strict=True, only=common)
    print(f"{CONFIRM:12s} majority vote (in more than half of {c['runs']} runs): precision {sv['precision']:.3f}  recall {sv['recall']:.3f}  "
          f"emitted {sv['emit']}  [separate number, not the iteration result]")
    print("\nStep 1b log:", f"precision {c['P']:.3f} (recall {c['R']:.3f});",
          "does NOT count (recall below floor)" if c["R"] < RECALL_FLOOR else "counts as iteration 1")
    print(f"  what changed: meta prompt v2 sample 0 ({WINNER}), written by {META_MODEL}; {SCREENED_TOTAL} candidates "
          "screened over 3 screens; picked by override (screening recall one element under the floor); new method; "
          "zero-shot (no worked examples); structural references include input ports the entity reads (F2); "
          "transaction records excluded (F1); sub-block connection fields are carriers (F3); bare names (F14); "
          "evidence levels (Established only)")
else:
    print("no confirmation runs yet")

no confirmation runs yet


## Refinement metrics: what to judge a prompt on

**Decide (the Step 1b rule).** These three decide whether to keep a prompt:
- **Precision:** the mean over 3 runs, strict matching, 15 modules.
- **Recall:** must stay ≥ 0.83.
- **F1.**

Keep a prompt if it adds at least 0.03 precision over the best so far. Everything below explains *why* a prompt scored as it did and *what to edit next*. None of it decides.

| Metric | What it shows | Edit it points to |
|---|---|---|
| **D1 Emission ratio** (emitted ÷ ground-truth entries) | How much the prompt over-reports | The rules for when to stop reporting |
| **D2 Precision by `realization`** (stores / sets / computes / exit port) | Which part of the structural-reference rule lets false positives in | e.g. low "exit port" precision means tighten the exit-port clause |
| **D3 Precision and recall by class** (port, signal, signal field; false positives per run) | Which kind of element goes wrong | e.g. field fan-out, bus ports |
| **D4 Concepts per module; elements per concept** | Too many concepts, or too many elements per concept (carriers counted as realizations) | The concept rules, or the structural rule |
| **D5 Validity** (missing modules, ungrounded names, wrong entity, bad values) | Format problems, which should be zero | Naming and schema instructions |
| **D6 Stability** (spread of precision and recall; Jaccard overlap of the reported sets across runs) | How consistently the method is followed; low overlap points to an ambiguous rule | Clarify the rule the unstable elements share |
| **D7 Worst modules by false positives; ground truth missed in most runs** | Where to look, and what recall is losing | Read those modules first; check what kind of element each miss is |
| **D8 False-positive families** (the 40-row CSV, grouped by hand) | The recurring error patterns | The critique for the next edit (ProTeGi step) |

Objective accuracy is recorded but not used for refinement at this stage. Known ground-truth gaps (e.g. `neorv32_cache` `inval_i`, `cache_o.cmd_dir`) appear in D7 for every prompt; don't chase them.

In [67]:
# Refinement metrics for the baseline, every screened candidate, and the confirmation (whichever exist)
_vs = [BASELINE] + [v for v in CANDIDATES if all((mt.run_dir(v, 0) / f"{s}.json").exists() for s, _ in MODULES)]
if WINNER and len(list(Path(".").glob(f"assets_tuning18_{WINNER}c_r*"))) >= 3:
    _vs.append(WINNER + "c")
_vs += [v for v in ("m7e194es0c",) if v not in _vs and mt.run_dir(v, 2).exists()]   # the seed, for comparison
_common, _rows, _n = mt.score_versions(_vs[1:], BASELINE)
DIAG = {v: mt.diagnostics(v, _common) for v in _vs}


== v2x3r8: 3 run(s), 15 modules, 111 ground-truth entries (exact)
   precision 0.296  recall 0.853
   D1_emit_ratio: 2.87987987987988
   D2_precision_by_realization: N/A
   D3_by_class: {'port': {'P': 0.372, 'R': 0.837, 'FP_per_run': 66.3}, 'signal': {'P': 0.347, 'R': 0.875, 'FP_per_run': 52.7}, 'signal-field': {'P': 0.209, 'R': 0.882, 'FP_per_run': 103.7}, 'absent': {'P': 0.0, 'R': 0.0, 'FP_per_run': 2.3}}
   D4_concepts_per_module: N/A
   D4_elements_per_concept: N/A
   D5_validity: {'modules_missing': 0}
   D6_stability: {'P_sd': 0.006, 'R_sd': 0.045, 'jaccard': 0.76}
   D7_worst_modules_by_FP: [('neorv32_cache', 32.0), ('neorv32_bus', 24.3), ('neorv32_spi', 23.7), ('neorv32_uart', 22.3), ('neorv32_twi', 20.7)]
   D7_missed_in_most_runs: ['neorv32_bus:keeper.halt (signal-field)', 'neorv32_bus:state (signal)', 'neorv32_cache:cache_o.cmd_dir (signal-field)', 'neorv32_cache:inval_i (absent)', 'neorv32_cpu:alu_res (signal)', 'neorv32_cpu:lsu_err (signal)', 'neorv32_cpu_cp_cfu:active_i 

## False positives to group by hand

This writes 40 random majority-vote false positives (seed 0) to `assetgen_meta/fp_sample_<version>.csv`, with everything filled in except `family` and `note`.

Line numbers are from `RTL_data/`, the files the model reads. For each row, pick the **first** family whose test matches:

1. **Bus boilerplate:** the standard bus request/response port, or a field of it.
2. **Reset / clock:** a clock, reset, clock enable or prescaler tick that only feeds timing or reset.
3. **Copy (relay):** its only assignment is `x <= y`, with no logic in between.
4. **Port plumbing / wrong side:** it only passes a value between a port and an internal signal, and the ground truth lists the other end.
5. **Twins:** one end of a connection across two entities, where the ground truth lists the other end.
6. **Decision-chain step:** a condition computed on the way to a ground-truth asset.
7. **Record-field fan-out:** a field of an internal record that the ground truth doesn't list.
8. **Other:** write one line on why.

In [68]:
if CONFIRM and len(list(Path(".").glob(f"assets_tuning18_{CONFIRM}_r*"))) >= 3:
    FP_CSV = Path("assetgen_meta") / f"fp_sample_{CONFIRM}.csv"
    n, total = mt.fp_sample(CONFIRM, FP_CSV, n=40, seed=0, baseline=BASELINE)
    print(f"wrote {n} of {total} majority-vote false positives to {FP_CSV}; fill in the 'family' column")
else:
    print("no confirmation runs yet")

no confirmation runs yet


## Hand arm m7e194es0ismr: Step 1b with the relationship map

**What changes.** The winner `m7e194es0ism` (precision 0.344, recall 0.847) gets ONE new section between its two frozen parts: the seed execution prompt (`e8df164fddb3`, the part that raised precision) and the worked examples (the part that restored recall). Both stay byte-identical; cell below checks it. The section teaches the executor to read the module's relationship map through the four IEEE P3164 questions (confidentiality, integrity, availability, undermined behaviour), and to tell a copy relay, a decision-chain input, a record-field fan-out element and a reset/clock driver from the element that realises the concept. The map is never a candidate list: the earlier relation arm `v2p3c1` failed exactly there (precision unchanged, recall -0.078, the list "became a menu").

**How it was made.** gpt-6-astra drafted the section with an analysis (how each P3164 question maps to relationship facts), critiqued it (8 issues, none high), then folded the positive reading procedure from its own analysis back in. One recorded post-edit: the stage name "Identify structural references". Everything is in `assetgen_meta/hand_arms/m7e194es0ismr/` (`exec_prompt.txt`, `relation_section.txt`, `build_info.json`, `astra/`).

**Input.** Comment-stripped RTL, then the block `=== RELATIONSHIP MAP ===` with a compact view of the module's map (`assetgen_meta/rel_view.py`: occurrence lists dropped, Occurrence IDs turned into RTL line numbers). The maps are experiment E3's (relation prompt v2 `8406f390b729`, run 1): **`step1/lasset_step1/relation_exp/E3_pairfirst_declaration/relation_map/8406f390b729/<module>.json`** (written by `lasset_step1.ipynb`, E3 cell). E3 is the most accurate writer measured (bake-off on an astra gold set, 2026-09-30: typed precision about 0.98, recall about 0.90, against E0's 0.93 / 0.83). The E0 Step 1 maps stay at `step1/lasset_step1/relation_map/a44eebd83bf7/<module>.json`. boot_rom, fifo and package have no map and get `(none for this module)`.

**Pre-registered.** Counts as an improvement over `m7e194es0ism` only with precision **0.374 or more** at recall at least **0.83** (the Step 1b rule: +0.03 precision). Below that, the map adds nothing to generation and `m7e194es0ism` stays the winner. Read also references per concept and concepts per run.

**What it breaks.** Nothing earlier: own folders `assets_tuning18_m7e194es0ismr_r0..r2`. Its input differs from every RTL-only arm. If the relation maps change (another prompt, or code-written relationships), this arm must be re-run on the new maps.

Run the **Setup** cell at the top first. The previous hand arm `m7e194es0ismp` stays re-runnable: set `HAND_VERSION = "m7e194es0ismp"` and `HAND_BUILD = None`.


In [2]:
# Hand arm: load the prompt and the relationship maps, and check both (stop here if a check fails)
import rel_view as rv
importlib.reload(rv)
HAND_VERSION  = "m7e194es0ismr"      # previous hand arm: "m7e194es0ismp" (RTL only: HAND_BUILD = None)
HAND_BUILD    = rv.build_user        # user message = RTL + "=== RELATIONSHIP MAP ===" block (None = RTL only)
HAND_DIR      = Path("assetgen_meta/hand_arms") / HAND_VERSION
HAND_PROMPT   = (HAND_DIR / "exec_prompt.txt").read_text(encoding="utf-8")
HAND_SEED     = Path("assetgen_meta/runs/meta_7e194e6254be_gpt-6-astra/sample_0/exec_prompt.txt").read_text(encoding="utf-8")
_BEST         = Path("assetgen_meta/hand_arms/m7e194es0ism/exec_prompt.txt").read_text(encoding="utf-8")
HAND_EXAMPLES = _BEST[len(HAND_SEED) + 2:]            # the winner's worked examples, frozen

# The relationship maps written by experiment E3 (lasset_step1.ipynb, E3 cell: relation_stage.merge):
#   step1/lasset_step1/relation_exp/E3_pairfirst_declaration/relation_map/8406f390b729/<module>.json   (8406f390b729 = relation prompt v2)
# (the E0 Step 1 maps: step1/lasset_step1/relation_map/a44eebd83bf7/<module>.json)
# The location is set in assetgen_meta/rel_view.py, line REL_MAP_DIR = ...; change it there to use another map.
REL_MAP_DIR = rv.REL_MAP_DIR                          # <-- where this arm reads the relationship maps

_chk  = mt.check_exec_prompt(HAND_PROMPT, mt.corpus_names(), HAND_VERSION)
_info = json.loads((HAND_DIR / "build_info.json").read_text(encoding="utf-8"))
_frozen = HAND_PROMPT.startswith(HAND_SEED + "\n\n") and HAND_PROMPT.endswith("\n\n" + HAND_EXAMPLES)
_block  = rv.BLOCK in HAND_PROMPT and rv.NONE_TEXT in HAND_PROMPT
print(f"{HAND_VERSION}: {len(HAND_PROMPT)} chars, sha {mt.sha12(HAND_PROMPT)} (build record {_info['system_sha12']}); "
      f"seed {_info['parts']['seed']['chars']} + map section {_info['parts']['relation_section']['chars']} "
      f"+ examples {_info['parts']['examples']['chars']}")
print("prompt checks:", "PASS" if _chk["ok"] else _chk["problems"])
print("seed and examples unchanged around the new section:", _frozen, "| map block named in the prompt:", _block)
print("relationship maps from", REL_MAP_DIR)
_MODULES = mt.rtl_modules()
_have = []
for _stem, _p in _MODULES:
    _mp = rv.map_path(_stem)
    _have.append(_mp.exists())
    print(f"   {_stem:24s} " + (f"map {_mp.stat().st_size:8,d} bytes -> compact {len(rv.map_text(_stem)):8,d} chars"
                                  if _mp.exists() else rv.NONE_TEXT))
print(f"maps found for {sum(_have)} of {len(_MODULES)} modules (the 15 test modules; boot_rom, fifo, package have none)")
assert _chk["ok"] and _frozen and _block and mt.sha12(HAND_PROMPT) == _info["system_sha12"], \
    "do not run: the prompt changed or a check failed"
assert sum(_have) == 15, "do not run: a relationship map is missing (see REL_MAP_DIR in assetgen_meta/rel_view.py)"


m7e194es0ismr: 114084 chars, sha fe430b81e2c1 (build record fe430b81e2c1); seed 26639 + map section 10404 + examples 77037
prompt checks: PASS
seed and examples unchanged around the new section: True | map block named in the prompt: True
relationship maps from E:\jobs\ff\test\Prepoison_subset\step1\lasset_step1\relation_exp\E3_pairfirst_declaration\relation_map\8406f390b729
   neorv32_boot_rom         (none for this module)
   neorv32_bus              map  723,076 bytes -> compact  281,310 chars
   neorv32_cache            map  140,566 bytes -> compact   46,855 chars
   neorv32_cpu              map   81,341 bytes -> compact   34,655 chars
   neorv32_cpu_cp_cfu       map   54,075 bytes -> compact   16,138 chars
   neorv32_cpu_cp_muldiv    map  121,415 bytes -> compact   36,580 chars
   neorv32_cpu_pmp          map   90,553 bytes -> compact   30,616 chars
   neorv32_debug_dtm        map  130,763 bytes -> compact   35,528 chars
   neorv32_fifo             (none for this module)
   neorv32

In [3]:
# Hand arm: 3 runs on the executor (gpt-5-mini, effort high, comment-stripped RTL + relationship map, no validation retry).
# Cached modules are skipped, so an interrupted run resumes. The three repeats run together; each prints when it ends.
from concurrent.futures import ThreadPoolExecutor, as_completed
_meta = {"stage": "hand arm: winner m7e194es0ism + relationship-map section (gpt-6-astra: draft, critique, revision)",
         "build": "assetgen_meta/hand_arms/m7e194es0ismr/build_info.json", "system_sha12": _info["system_sha12"],
         "relation_maps": str(REL_MAP_DIR), "map_view": "assetgen_meta/rel_view.py compact"}
_note = "comment-stripped RTL + relationship map (compact)" if HAND_BUILD else "comment-stripped RTL only"
with ThreadPoolExecutor(max_workers=3) as _pool:
    _futs = {_pool.submit(mt.run_version, client, HAND_VERSION, _rep, HAND_PROMPT, _MODULES,
                          model="gpt-5-mini", extra_meta=_meta, workers=6,
                          build=HAND_BUILD, input_note=_note): _rep for _rep in range(3)}
    for _f in as_completed(_futs):
        _rep = _futs[_f]
        _s = _f.result()
        print(f"{HAND_VERSION} r{_rep}: {_s['modules_done']}/{len(_MODULES)} modules, tokens {_s['tokens']}, "
              f"validation issues {_s['issues']} {_s['issue_kinds']}")
        for _k, _n in _s["notes"].items():
            if _n not in ("ok", "cached"):
                print("   ", _k, _n)


m7e194es0ismr r1: 18/18 modules, tokens {'in': 898532, 'out': 284214}, validation issues 1 {'ungrounded': 1}
m7e194es0ismr r2: 18/18 modules, tokens {'in': 898532, 'out': 288310}, validation issues 1 {'ungrounded': 1}
m7e194es0ismr r0: 18/18 modules, tokens {'in': 898532, 'out': 300618}, validation issues 1 {'ungrounded': 1}


In [4]:
# Hand arm: score it (pre-registered: precision >= 0.374 at recall >= 0.83 against m7e194es0ism)
sys.path.insert(0, "assetgen_meta/hand_arms")
import score_hand_arm as sha
importlib.reload(sha)
_ = sha.score(HAND_VERSION, refs=("v2x3r8", "m7e194es0c", "m7e194es0ism"))


15 modules, 111 ground-truth entries (exact); runs {'v2x3r8': 3, 'm7e194es0c': 3, 'm7e194es0ism': 3, 'm7e194es0ismr': 3}

1) precision / recall / items per run (strict)
   v2x3r8         P 0.296 R 0.853 items  319.7 (2.88x GT) | per run P [0.289, 0.303, 0.295] R [0.793, 0.901, 0.865] items [304, 330, 325]
   m7e194es0c     P 0.379 R 0.787 items  230.3 (2.08x GT) | per run P [0.383, 0.388, 0.367] R [0.793, 0.811, 0.757] items [230, 232, 229]
   m7e194es0ism   P 0.344 R 0.847 items  273.3 (2.46x GT) | per run P [0.347, 0.332, 0.353] R [0.856, 0.82, 0.865] items [274, 274, 272]
   m7e194es0ismr  P 0.353 R 0.778 items  244.7 (2.20x GT) | per run P [0.356, 0.355, 0.347] R [0.793, 0.784, 0.757] items [247, 245, 242]

2) STEP 1b VERDICT: recall 0.778 < 0.83; precision 0.353 >= 0.326 -> does NOT count

3) exact permutation test against the seed (3 vs 3; p = 0.10 is the lowest possible)
   precision seed [0.383, 0.388, 0.367] new [0.356, 0.355, 0.347] diff -0.026 p=0.10
   recall    seed [0.793

## Hand arm m7e194es0ismc: the same prompt, code-written relationship map

Run after `m7e194es0ismr`. **One change:** the relationship map. Same prompt file (`exec_prompt.txt`, sha checked below), same executor and settings; the map's `relationship` records are written by code (`step1/code_pairs_v2.py`, frozen sha `b0e767000ec2`), its `functionality` sentences come from E3 run 1, and every program field is written by `relation_stage.merge` as before. Code-written relationships were the most accurate measured (bake-off on the held-out astra gold set, 2026-09-30: typed precision 0.988, recall 0.988, against E3's about 0.98 / 0.90). The code writes no `bits`.

**Maps:** `step1/lasset_step1/relation_map_code/b0e767000ec2_func-8406f390b729/<module>.json` (built by `build_code_map_arm.py`; location set in `assetgen_meta/rel_view.py`, `CODE_MAP_DIR`).

**Read against `m7e194es0ismr`** (the pre-registered bar stays: precision 0.374 or more at recall at least 0.83 against `m7e194es0ism`). A difference between ismr and ismc is the effect of the map's relationship accuracy; with 3 runs each, small differences are noise.


In [10]:
# Hand arm ismc: load and check (same prompt as ismr, code-written maps)
import rel_view as rv
importlib.reload(rv)
HAND_VERSION_C = "m7e194es0ismc"
HAND_DIR_C     = Path("assetgen_meta/hand_arms") / HAND_VERSION_C
HAND_PROMPT_C  = (HAND_DIR_C / "exec_prompt.txt").read_text(encoding="utf-8")
# The code-written relationship maps (relationship: step1/code_pairs_v2.py, functionality: E3 run 1):
#   step1/lasset_step1/relation_map_code/b0e767000ec2_func-8406f390b729/<module>.json
CODE_MAP_DIR   = rv.CODE_MAP_DIR                      # <-- where this arm reads the relationship maps (rel_view.py)
HAND_BUILD_C   = rv.make_build(CODE_MAP_DIR)
_info_c = json.loads((HAND_DIR_C / "build_info.json").read_text(encoding="utf-8"))
_chk_c  = mt.check_exec_prompt(HAND_PROMPT_C, mt.corpus_names(), HAND_VERSION_C)
_MODULES = mt.rtl_modules()
_have_c = [rv.map_path(s, CODE_MAP_DIR).exists() for s, _p in _MODULES]
print(f"{HAND_VERSION_C}: sha {mt.sha12(HAND_PROMPT_C)} (ismr build record {_info_c['system_sha12']}); prompt checks:",
      "PASS" if _chk_c["ok"] else _chk_c["problems"])
print(f"maps from {CODE_MAP_DIR}: {sum(_have_c)} of {len(_MODULES)} modules")
assert _chk_c["ok"] and mt.sha12(HAND_PROMPT_C) == _info_c["system_sha12"] and sum(_have_c) == 15, "do not run"


m7e194es0ismc: sha fe430b81e2c1 (ismr build record fe430b81e2c1); prompt checks: PASS
maps from E:\jobs\ff\test\Prepoison_subset\step1\lasset_step1\relation_map_code\b0e767000ec2_func-8406f390b729: 15 of 18 modules


In [11]:
# Hand arm ismc: 3 runs on the executor (gpt-5-mini, effort high, comment-stripped RTL + code-written map)
from concurrent.futures import ThreadPoolExecutor, as_completed
_meta_c = {"stage": "hand arm: ismr prompt + code-written relationship map", "build": str(HAND_DIR_C / "build_info.json"),
           "system_sha12": _info_c["system_sha12"], "relation_maps": str(CODE_MAP_DIR), "map_view": "assetgen_meta/rel_view.py compact"}
with ThreadPoolExecutor(max_workers=3) as _pool:
    _futs = {_pool.submit(mt.run_version, client, HAND_VERSION_C, _rep, HAND_PROMPT_C, _MODULES, model="gpt-5-mini",
                          extra_meta=_meta_c, workers=6, build=HAND_BUILD_C,
                          input_note="comment-stripped RTL + relationship map (code-written, compact)"): _rep for _rep in range(3)}
    for _f in as_completed(_futs):
        _s = _f.result()
        print(f"{HAND_VERSION_C} r{_futs[_f]}: {_s['modules_done']}/{len(_MODULES)} modules, tokens {_s['tokens']}, "
              f"validation issues {_s['issues']} {_s['issue_kinds']}")
        for _k, _n in _s["notes"].items():
            if _n not in ("ok", "cached"):
                print("   ", _k, _n)


m7e194es0ismc r2: 18/18 modules, tokens {'in': 236551, 'out': 81622}, validation issues 2 {'ungrounded': 2}
m7e194es0ismc r1: 18/18 modules, tokens {'in': 288705, 'out': 111348}, validation issues 1 {'ungrounded': 1}
m7e194es0ismc r0: 18/18 modules, tokens {'in': 288705, 'out': 107467}, validation issues 0 {}


In [12]:
# Hand arm ismc: score it, next to ismr (same prompt, E3 maps)
sys.path.insert(0, "assetgen_meta/hand_arms")
import score_hand_arm as sha
importlib.reload(sha)
_ = sha.score(HAND_VERSION_C, refs=("v2x3r8", "m7e194es0ism", "m7e194es0ismr"))


15 modules, 111 ground-truth entries (exact); runs {'v2x3r8': 3, 'm7e194es0ism': 3, 'm7e194es0ismr': 3, 'm7e194es0ismc': 3}

1) precision / recall / items per run (strict)
   v2x3r8         P 0.296 R 0.853 items  319.7 (2.88x GT) | per run P [0.289, 0.303, 0.295] R [0.793, 0.901, 0.865] items [304, 330, 325]
   m7e194es0ism   P 0.344 R 0.847 items  273.3 (2.46x GT) | per run P [0.347, 0.332, 0.353] R [0.856, 0.82, 0.865] items [274, 274, 272]
   m7e194es0ismr  P 0.353 R 0.778 items  244.7 (2.20x GT) | per run P [0.356, 0.355, 0.347] R [0.793, 0.784, 0.757] items [247, 245, 242]
   m7e194es0ismc  P 0.346 R 0.787 items  252.3 (2.27x GT) | per run P [0.357, 0.348, 0.332] R [0.82, 0.829, 0.712] items [255, 264, 238]

2) STEP 1b VERDICT: recall 0.787 < 0.83; precision 0.346 >= 0.326 -> does NOT count

4) ground-truth hits and false positives per run, by element kind
   kind                               GT | hits/run       v2x3r8 m7e194es0ism m7e194es0ism m7e194es0ism | FP/run       v2x3r8 

## Hand arm m7e194es0ismcap: the winner plus the captured-input rule (RTL only)

**One change.** The winner `m7e194es0ism` with one bullet inserted after seed line 218 (`build_info.json` has the text). It narrows the seed's "unchanged forwarding" exclusion: capturing an input's value into a storing element of this entity is storage, so the input is a setting point of the concept that element stores. Clock, clock-enable, reset and transport inputs stay excluded. No relationship map in the input; the run is directly comparable with the winner.

**Why.** The winner forms the concept of a sampled input but credits only the register that samples it (`uart_rxd_i`, `spi_dat_i`, `jtag_tms_i`, `enable_i` ...). Basis: seed L13 and L57, IEEE P3164 3.1.2. Reviewed by two Claude agents (2026-10-01): "report the register too" was dropped (+6.7 FP per run, 0 GT), the exclusions were restated (otherwise 3 clock/reset pins and 19 transport ports would be admitted). Every count was re-derived independently.

**Expected (mechanical bound, not a model run):** +3.3 to +4 GT and +1.7 non-GT per run: recall about 0.85 -> 0.88, precision about flat.

**Pre-registered readings** (`hand_arms/read_cap_arm.py`, fixed before the run): the 4 target inputs are at 2 of 12 run-slots in the winner; the rule is **falsified if they reach 4 or fewer of 12**. Guards: clock/reset-role inputs >= 3 of 9, transport ports >= 3, first-stage pin registers >= 3 of 15, pmp `ctrl_i` 0 of 3, or emitted per run well above the winner's 273 mean an exclusion slipped or the bullet became a quota. The Step 1b bar still applies for adoption: precision >= 0.374 at recall >= 0.83 against the winner is not expected here; the arm is a recall repair to be read on its readings.

Run the **Setup** cell first. Needs OpenAI credit (about 54 calls, gpt-5-mini).


In [2]:
# Hand arm ismcap: load and check (winner + one bullet, RTL only)
HAND_VERSION_K = "m7e194es0ismcap"
HAND_DIR_K     = Path("assetgen_meta/hand_arms") / HAND_VERSION_K
HAND_PROMPT_K  = (HAND_DIR_K / "exec_prompt.txt").read_text(encoding="utf-8")
_info_k = json.loads((HAND_DIR_K / "build_info.json").read_text(encoding="utf-8"))
_chk_k  = mt.check_exec_prompt(HAND_PROMPT_K, mt.corpus_names(), HAND_VERSION_K)
_win    = Path("assetgen_meta/hand_arms/m7e194es0ism/exec_prompt.txt").read_text(encoding="utf-8")
import difflib
_diff = [l for l in difflib.unified_diff(_win.split("\n"), HAND_PROMPT_K.split("\n"), lineterm="", n=0) if l[:1] in "+-" and l[:3] not in ("+++", "---")]
print(f"{HAND_VERSION_K}: sha {mt.sha12(HAND_PROMPT_K)} (build record {_info_k['system_sha12']}); prompt checks:", "PASS" if _chk_k["ok"] else _chk_k["problems"])
print("difference from the winner:", _diff)
_MODULES = mt.rtl_modules()
assert _chk_k["ok"] and mt.sha12(HAND_PROMPT_K) == _info_k["system_sha12"] and len(_diff) == 1 and _diff[0].startswith("+- Capture"), "do not run"


m7e194es0ismcap: sha e1c7d09ba571 (build record e1c7d09ba571); prompt checks: PASS
difference from the winner: ["+- Capture of an input's value into a storing element of this entity, directly or through this entity's register stages, is storage, not unchanged forwarding: the input is a setting point of the concept that element stores. Clock, clock-enable, reset and transport inputs remain excluded."]


In [3]:
# Hand arm ismcap: 3 runs on the executor (gpt-5-mini, effort high, comment-stripped RTL only, no validation retry)
from concurrent.futures import ThreadPoolExecutor, as_completed
_meta_k = {"stage": "hand arm: winner + captured-input bullet after seed L218", "build": str(HAND_DIR_K / "build_info.json"),
           "system_sha12": _info_k["system_sha12"]}
with ThreadPoolExecutor(max_workers=3) as _pool:
    _futs = {_pool.submit(mt.run_version, client, HAND_VERSION_K, _rep, HAND_PROMPT_K, _MODULES, model="gpt-5-mini",
                          extra_meta=_meta_k, workers=6): _rep for _rep in range(3)}
    for _f in as_completed(_futs):
        _s = _f.result()
        print(f"{HAND_VERSION_K} r{_futs[_f]}: {_s['modules_done']}/{len(_MODULES)} modules, tokens {_s['tokens']}, "
              f"validation issues {_s['issues']} {_s['issue_kinds']}")
        for _k, _n in _s["notes"].items():
            if _n not in ("ok", "cached"):
                print("   ", _k, _n)


m7e194es0ismcap r2: 18/18 modules, tokens {'in': 670685, 'out': 302766}, validation issues 0 {}
m7e194es0ismcap r1: 18/18 modules, tokens {'in': 670685, 'out': 325745}, validation issues 1 {'ungrounded': 1}
m7e194es0ismcap r0: 18/18 modules, tokens {'in': 670685, 'out': 327305}, validation issues 5 {'ungrounded': 5}


In [4]:
# Hand arm ismcap: score it, then the pre-registered readings against the winner
sys.path.insert(0, "assetgen_meta/hand_arms")
import score_hand_arm as sha, read_cap_arm as rca
importlib.reload(sha); importlib.reload(rca)
_ = sha.score(HAND_VERSION_K, refs=("v2x3r8", "m7e194es0ism"))
rca.read(HAND_VERSION_K, "m7e194es0ism")


15 modules, 111 ground-truth entries (exact); runs {'v2x3r8': 3, 'm7e194es0ism': 3, 'm7e194es0ismcap': 3}

1) precision / recall / items per run (strict)
   v2x3r8         P 0.296 R 0.853 items  319.7 (2.88x GT) | per run P [0.289, 0.303, 0.295] R [0.793, 0.901, 0.865] items [304, 330, 325]
   m7e194es0ism   P 0.344 R 0.847 items  273.3 (2.46x GT) | per run P [0.347, 0.332, 0.353] R [0.856, 0.82, 0.865] items [274, 274, 272]
   m7e194es0ismcap P 0.355 R 0.829 items  259.3 (2.34x GT) | per run P [0.348, 0.359, 0.357] R [0.838, 0.829, 0.82] items [267, 256, 255]

2) STEP 1b VERDICT: recall 0.829 < 0.83; precision 0.355 >= 0.326 -> does NOT count

4) ground-truth hits and false positives per run, by element kind
   kind                               GT | hits/run       v2x3r8 m7e194es0ism m7e194es0ism | FP/run       v2x3r8 m7e194es0ism m7e194es0ism
   absent from closed set              1 |          0.0          0.0          0.0 |                 2.3          1.0          0.7
   clock/res

## Hand arms m7e194es0ismq and m7e194es0ismrq: ask the four questions, require a negative

**One change, applied to both existing prompts.** The audit (2026-10-01) found that the model's reasoning never answers the IEEE P3164 questions: 315 of 318 concepts said "Integrity", no concept ever answered a question in the negative, and one argument ("the RTL uses this value, so corrupting it breaks behavior") served every element. These arms add, inside stage d of the seed, the four questions (confidentiality, integrity, availability, undermined behavior), each answered yes or no with a quoted RTL statement; a value with no yes is not an asset and goes to a new top-level `excluded values` list; the two contract sentences that forbade excluded cases are re-routed to that list; each concept gains `security questions`. Everything else is byte-identical to the base:

- `m7e194es0ismq` = winner `m7e194es0ism` + questions, RTL-only input. Read against the winner.
- `m7e194es0ismrq` = map arm `m7e194es0ismr` + the same questions, E3 relation map in the input. Read against `ismr`.

**Known risk, from the project log.** `v2sec` added an output destination for set-aside elements and recall fell 0.925 -> 0.628 (95% of departures moved there); `T-1` changed the schema and emissions rose 28%. The readings below catch both.

**Pre-registered readings** (`hand_arms/read_q_arms.py`): compliance (all four answers present; yes-statements verbatim in the RTL), the share of no answers per question, the objective mix, excluded values per run and how many name a ground-truth asset (fail if more than 1 per run), emissions per run, and P/R against the base. Counts as an improvement only if P or R do not fall below the base by more than 0.03 (run noise) and the negatives are real (verbatim statements). The scorer ignores the new fields, so P/R stay comparable.

Run the **Setup** cell first. Each arm is 3 runs x 18 modules, about 54 calls of gpt-5-mini.


In [6]:
# Four-question arms: load and check both prompts (each differs from its base only by the question edit)
import rel_view as rv
importlib.reload(rv)
import difflib
Q_ARMS = {"m7e194es0ismq": ("m7e194es0ism", None),                 # winner + questions, RTL only
          "m7e194es0ismrq": ("m7e194es0ismr", rv.build_user)}       # map arm + questions, E3 map input (rel_view.REL_MAP_DIR)
Q_PROMPT, Q_INFO = {}, {}
for _arm, (_base, _b) in Q_ARMS.items():
    _d = Path("assetgen_meta/hand_arms") / _arm
    Q_PROMPT[_arm] = (_d / "exec_prompt.txt").read_text(encoding="utf-8")
    Q_INFO[_arm] = json.loads((_d / "build_info.json").read_text(encoding="utf-8"))
    _bt = (Path("assetgen_meta/hand_arms") / _base / "exec_prompt.txt").read_text(encoding="utf-8")
    _chk = mt.check_exec_prompt(Q_PROMPT[_arm], mt.corpus_names(), _arm)
    _diff = [l for l in difflib.unified_diff(_bt.split("\n"), Q_PROMPT[_arm].split("\n"), lineterm="", n=0) if l[:1] in "+-" and l[:3] not in ("+++", "---")]
    print(f"{_arm}: sha {mt.sha12(Q_PROMPT[_arm])} (build record {Q_INFO[_arm]['system_sha12']}); checks:",
          "PASS" if _chk["ok"] else _chk["problems"], f"| diff vs {_base}: -{sum(l[0] == '-' for l in _diff)}/+{sum(l[0] == '+' for l in _diff)} lines")
    assert _chk["ok"] and mt.sha12(Q_PROMPT[_arm]) == Q_INFO[_arm]["system_sha12"], "do not run"
_MODULES = mt.rtl_modules()
print("E3 maps for ismrq:", sum(rv.map_path(s).exists() for s, _p in _MODULES), "of", len(_MODULES), "modules, from", rv.REL_MAP_DIR)


m7e194es0ismq: sha 4629c6a2e3a2 (build record 4629c6a2e3a2); checks: PASS | diff vs m7e194es0ism: -4/+32 lines
m7e194es0ismrq: sha 24a1ac5f1f71 (build record 24a1ac5f1f71); checks: PASS | diff vs m7e194es0ismr: -4/+32 lines
E3 maps for ismrq: 15 of 18 modules, from E:\jobs\ff\test\Prepoison_subset\step1\lasset_step1\relation_exp\E3_pairfirst_declaration\relation_map\8406f390b729


In [8]:
# Four-question arms: 3 runs each on the executor (gpt-5-mini, effort high). Cached modules are skipped.
from concurrent.futures import ThreadPoolExecutor, as_completed
for _arm, (_base, _b) in Q_ARMS.items():
    _meta = {"stage": f"hand arm: {_base} + four P3164 questions with a required negative", "system_sha12": Q_INFO[_arm]["system_sha12"],
             "build": str(Path("assetgen_meta/hand_arms") / _arm / "build_info.json")}
    with ThreadPoolExecutor(max_workers=3) as _pool:
        _futs = {_pool.submit(mt.run_version, client, _arm, _rep, Q_PROMPT[_arm], _MODULES, model="gpt-5-mini", extra_meta=_meta,
                              workers=6, build=_b, input_note=Q_INFO[_arm]["input"]): _rep for _rep in range(3)}
        for _f in as_completed(_futs):
            _s = _f.result()
            print(f"{_arm} r{_futs[_f]}: {_s['modules_done']}/{len(_MODULES)} modules, tokens {_s['tokens']}, validation issues {_s['issues']} {_s['issue_kinds']}")
            for _k, _n in _s["notes"].items():
                if _n not in ("ok", "cached"):
                    print("   ", _k, _n)


m7e194es0ismq r1: 18/18 modules, tokens {'in': 0, 'out': 0}, validation issues 3 {'ungrounded': 3}
m7e194es0ismq r0: 18/18 modules, tokens {'in': 0, 'out': 0}, validation issues 3 {'ungrounded': 3}
m7e194es0ismq r2: 18/18 modules, tokens {'in': 0, 'out': 0}, validation issues 1 {'ungrounded': 1}
m7e194es0ismrq r1: 18/18 modules, tokens {'in': 0, 'out': 0}, validation issues 2 {'ungrounded': 2}
m7e194es0ismrq r2: 18/18 modules, tokens {'in': 42028, 'out': 25353}, validation issues 2 {'ungrounded': 2}
m7e194es0ismrq r0: 18/18 modules, tokens {'in': 42570, 'out': 29707}, validation issues 2 {'ungrounded': 2}


In [9]:
# Four-question arms: score against the bases, then the pre-registered readings
sys.path.insert(0, "assetgen_meta/hand_arms")
import score_hand_arm as sha, read_q_arms as rqa
importlib.reload(sha); importlib.reload(rqa)
_ = sha.score("m7e194es0ismq", refs=("v2x3r8", "m7e194es0ism"))
_ = sha.score("m7e194es0ismrq", refs=("v2x3r8", "m7e194es0ism", "m7e194es0ismr"))
for _arm in Q_ARMS:
    rqa.read(_arm); print()


15 modules, 111 ground-truth entries (exact); runs {'v2x3r8': 3, 'm7e194es0ism': 3, 'm7e194es0ismq': 3}

1) precision / recall / items per run (strict)
   v2x3r8         P 0.296 R 0.853 items  319.7 (2.88x GT) | per run P [0.289, 0.303, 0.295] R [0.793, 0.901, 0.865] items [304, 330, 325]
   m7e194es0ism   P 0.344 R 0.847 items  273.3 (2.46x GT) | per run P [0.347, 0.332, 0.353] R [0.856, 0.82, 0.865] items [274, 274, 272]
   m7e194es0ismq  P 0.352 R 0.754 items  237.7 (2.14x GT) | per run P [0.378, 0.351, 0.328] R [0.793, 0.775, 0.694] items [233, 245, 235]

2) STEP 1b VERDICT: recall 0.754 < 0.83; precision 0.352 >= 0.326 -> does NOT count

4) ground-truth hits and false positives per run, by element kind
   kind                               GT | hits/run       v2x3r8 m7e194es0ism m7e194es0ism | FP/run       v2x3r8 m7e194es0ism m7e194es0ism
   absent from closed set              1 |          0.0          0.0          0.0 |                 2.3          1.0          0.7
   clock/reset

## Hand arm m7e194es0ismd: the definition rules (ASSET_DEFINITION.md)

**One change class, from `assetgen_meta/ASSET_DEFINITION.md` section 3.** The winner `m7e194es0ism` with four seed lines amended; the worked examples are byte-identical:
1. A state, setting or decision element realises its own concept and is primary in its own right (basis: SA-EDI "Control" type, IEEE P3164 questions 2-4, Nath & Tan's control/configuration/status patterns). This resolves the seed's lines 15 and 97, which treated every influencer as secondary.
2. A register that holds a value only while moving it from where it enters the module to where it leaves carries that value: it is secondary (basis: LAsset p2, whose AES output buffer is a secondary asset; SAIF p2).

Input is RTL only, so it is compared directly with the winner.

**Pre-registered readings** (`hand_arms/read_def_arm.py`, fixed before the run): R1 false positives on a frozen list of 17 transit registers and FIFO data fields (33 slots in the winner) must fall by at least half; R2 reference hits on deciding elements must not fall; R3 five references the transit rule must not drop (imem `rdata`, muldiv `mul.prod`, cfu `key_mem`, hwspinlock `lock_q`, uart `ctrl.baud`) stay in at least 2 of 3 runs; R4 the false-positive price of rule 1 on deciding elements; R5 emissions within 10% of the winner's 273 per run. Every row prints next to **LAsset initial** (no CWE refinement): P 0.737, R 0.910, 137 emitted on these 15 modules.

Run the **Setup** cell first. 3 runs x 18 modules, about 54 calls of gpt-5-mini.


In [14]:
# Definition arm ismd: load and check (winner + four amended seed lines, RTL only)
import difflib
HAND_VERSION_D = "m7e194es0ismd"
HAND_DIR_D     = Path("assetgen_meta/hand_arms") / HAND_VERSION_D
HAND_PROMPT_D  = (HAND_DIR_D / "exec_prompt.txt").read_text(encoding="utf-8")
_info_d = json.loads((HAND_DIR_D / "build_info.json").read_text(encoding="utf-8"))
_chk_d  = mt.check_exec_prompt(HAND_PROMPT_D, mt.corpus_names(), HAND_VERSION_D)
_win    = Path("assetgen_meta/hand_arms/m7e194es0ism/exec_prompt.txt").read_text(encoding="utf-8")
_diff_d = [l for l in difflib.unified_diff(_win.split("\n"), HAND_PROMPT_D.split("\n"), lineterm="", n=0) if l[:1] in "+-" and l[:3] not in ("+++", "---")]
print(f"{HAND_VERSION_D}: sha {mt.sha12(HAND_PROMPT_D)} (build record {_info_d['system_sha12']}); checks:", "PASS" if _chk_d["ok"] else _chk_d["problems"],
      f"| lines changed vs the winner: -{sum(l[0] == '-' for l in _diff_d)}/+{sum(l[0] == '+' for l in _diff_d)}")
_MODULES = mt.rtl_modules()
assert _chk_d["ok"] and mt.sha12(HAND_PROMPT_D) == _info_d["system_sha12"] and len(_diff_d) == 8, "do not run"


m7e194es0ismd: sha 6f74d32530ab (build record 6f74d32530ab); checks: PASS | lines changed vs the winner: -4/+4


In [15]:
# Definition arm ismd: 3 runs on the executor (gpt-5-mini, effort high, comment-stripped RTL only). Cached modules are skipped.
from concurrent.futures import ThreadPoolExecutor, as_completed
_meta_d = {"stage": "hand arm: winner + ASSET_DEFINITION.md rules (deciders primary; transit registers secondary)",
           "build": str(HAND_DIR_D / "build_info.json"), "system_sha12": _info_d["system_sha12"]}
with ThreadPoolExecutor(max_workers=3) as _pool:
    _futs = {_pool.submit(mt.run_version, client, HAND_VERSION_D, _rep, HAND_PROMPT_D, _MODULES, model="gpt-5-mini",
                          extra_meta=_meta_d, workers=6): _rep for _rep in range(3)}
    for _f in as_completed(_futs):
        _s = _f.result()
        print(f"{HAND_VERSION_D} r{_futs[_f]}: {_s['modules_done']}/{len(_MODULES)} modules, tokens {_s['tokens']}, "
              f"validation issues {_s['issues']} {_s['issue_kinds']}")
        for _k, _n in _s["notes"].items():
            if _n not in ("ok", "cached"):
                print("   ", _k, _n)


m7e194es0ismd r1: 18/18 modules, tokens {'in': 671927, 'out': 302305}, validation issues 2 {'ungrounded': 2}
m7e194es0ismd r0: 18/18 modules, tokens {'in': 671927, 'out': 334564}, validation issues 9 {'ungrounded': 9}
m7e194es0ismd r2: 18/18 modules, tokens {'in': 671927, 'out': 335044}, validation issues 2 {'ungrounded': 2}


In [16]:
# Definition arm ismd: score against the winner, then the pre-registered readings (with the LAsset initial row)
sys.path.insert(0, "assetgen_meta/hand_arms")
import score_hand_arm as sha, read_def_arm as rda
importlib.reload(sha); importlib.reload(rda)
_ = sha.score(HAND_VERSION_D, refs=("v2x3r8", "m7e194es0ism"))
rda.read(HAND_VERSION_D, "m7e194es0ism")


15 modules, 111 ground-truth entries (exact); runs {'v2x3r8': 3, 'm7e194es0ism': 3, 'm7e194es0ismd': 3}

1) precision / recall / items per run (strict)
   v2x3r8         P 0.296 R 0.853 items  319.7 (2.88x GT) | per run P [0.289, 0.303, 0.295] R [0.793, 0.901, 0.865] items [304, 330, 325]
   m7e194es0ism   P 0.344 R 0.847 items  273.3 (2.46x GT) | per run P [0.347, 0.332, 0.353] R [0.856, 0.82, 0.865] items [274, 274, 272]
   m7e194es0ismd  P 0.340 R 0.877 items  286.3 (2.58x GT) | per run P [0.336, 0.355, 0.33] R [0.874, 0.883, 0.874] items [289, 276, 294]

2) STEP 1b VERDICT: recall 0.877 >= 0.83; precision 0.340 >= 0.326 -> COUNTS

4) ground-truth hits and false positives per run, by element kind
   kind                               GT | hits/run       v2x3r8 m7e194es0ism m7e194es0ism | FP/run       v2x3r8 m7e194es0ism m7e194es0ism
   absent from closed set              1 |          0.0          0.0          0.0 |                 2.3          1.0          1.0
   clock/reset input  

## Held-out validation (run ONCE): the winner + frozen code levers on the 26 never-used modules

**What it tests.** Whether the post-generation code levers measured on the 15 tuning modules generalise: majority vote (MV), dropping elements with no evidence path (NONE), and back-filling captured input pins (BF). On tuning, MV+NONE+BF gives P 0.394 / R 0.901 against the winner's 0.344 / 0.847. The prompt is the winner `m7e194es0ism`, unchanged (sha `0d0def4c6fe3`).

**Pre-registered** in `assetgen_meta/HELDOUT_PREREG.md`, which pins the sha of 18 files; `heldout_readings.read()` refuses to score if any of them changed, if a run folder used another prompt, model or closed set, or if the readings were already taken once.
- **R1 (decides):** MV+NONE+BF is adopted if its precision gain over MV has a module-bootstrap 95% interval above zero, and its recall is at least 0.83.
- **R2:** GUARD only as a secondary row, against random thinning of equal strength (it is overfitted, see ASSET_DEFINITION.md).
- **R3-R6:** each filter alone, the leak checks, base P and R per run.
- Every row prints next to **LAsset initial** (no CWE refinement) on the same modules: P 0.734, R 0.862, 222 emitted over the 26 modules.

**What is not blind.** The clock/reset and bus-transaction conventions were checked on all 41 modules early on, so absolute held-out precision is optimistic; the clean test is the gain each filter adds on the same runs.

**Everything except the executor is code.** The relation maps for the 26 modules were built without any model (a code SITE tagger that matches the LLM profiler on 5,734 of 5,760 tuning rows): `step1/lasset_step1/relation_map_code_heldout/b0e767000ec2_codetags/`. Closed sets: `parsed_heldout26/`.

**Cost:** 78 calls of gpt-5-mini (3 runs x 26 modules), estimated $2.80-$4.06. Run the **Setup** cell first, then these three cells in order. Do not edit any pinned file between the pre-registration and the reading.


In [17]:
# Held-out: check the pre-registration and the inputs (no API call)
sys.path.insert(0, "assetgen_meta")
import heldout_readings as HR
importlib.reload(HR)
HELD_PROMPT = Path("assetgen_meta/hand_arms/m7e194es0ism/exec_prompt.txt").read_text(encoding="utf-8")
HELD_MODULES = mt.rtl_modules("RTL_heldout")
assert HR.verify_prereg(), "a pinned file changed since the pre-registration: do not run"
assert mt.sha12(HELD_PROMPT) == "0d0def4c6fe3", "the winner prompt changed"
_maps = Path("step1/lasset_step1/relation_map_code_heldout/b0e767000ec2_codetags")
print(f"held-out modules {len(HELD_MODULES)}; maps {sum((_maps / f'{s}.json').exists() for s, _p in HELD_MODULES)}; "
      f"closed sets {sum(Path('parsed_heldout26', f'{s}.json').exists() for s, _p in HELD_MODULES)}")
assert len(HELD_MODULES) == 26


pre-registered files: 18/18 unchanged
held-out modules 26; maps 26; closed sets 26


In [18]:
# Held-out: 3 runs of the winner on the 26 held-out modules (gpt-5-mini, effort high, comment-stripped RTL only).
# 78 calls. Cached modules are skipped, so an interrupted run resumes; re-run this cell for any module that failed.
from concurrent.futures import ThreadPoolExecutor, as_completed
_meta_h = {"stage": "held-out validation: winner m7e194es0ism, pre-registered in assetgen_meta/HELDOUT_PREREG.md", "system_sha12": "0d0def4c6fe3"}
with ThreadPoolExecutor(max_workers=3) as _pool:
    _futs = {_pool.submit(mt.run_version, client, "m7e194es0ism", _rep, HELD_PROMPT, HELD_MODULES, model="gpt-5-mini",
                          extra_meta=_meta_h, workers=6, stem="assets_heldout26", parsed_dir="parsed_heldout26"): _rep for _rep in range(3)}
    for _f in as_completed(_futs):
        _s = _f.result()
        print(f"held-out r{_futs[_f]}: {_s['modules_done']}/{len(HELD_MODULES)} modules, tokens {_s['tokens']}, "
              f"validation issues {_s['issues']} {_s['issue_kinds']}")
        for _k, _n in _s["notes"].items():
            if _n not in ("ok", "cached"):
                print("   ", _k, _n)


held-out r0: 26/26 modules, tokens {'in': 983875, 'out': 449701}, validation issues 14 {'ungrounded': 14}
held-out r2: 26/26 modules, tokens {'in': 983875, 'out': 467572}, validation issues 18 {'ungrounded': 18}
held-out r1: 26/26 modules, tokens {'in': 983875, 'out': 483431}, validation issues 24 {'ungrounded': 24}


In [19]:
# Held-out: the pre-registered readings, taken ONCE (no API call). Refuses if anything pinned changed or if already read.
importlib.reload(HR)
HELD_READINGS = HR.read()


pre-registered files: 18/18 unchanged
post_levers self-test PASS: 17 stacks and the LAsset initial row reproduce to 3 decimals (15 tuning modules, 111 GT entries per run, code-tag maps and conditions)
   assets_heldout26_m7e194es0ism_r0: 26/26 modules; missing none; OK
   assets_heldout26_m7e194es0ism_r1: 26/26 modules; missing none; OK
   assets_heldout26_m7e194es0ism_r2: 26/26 modules; missing none; OK
   traces r0: 521 records -> assetgen_meta\traces\assets_heldout26\m7e194es0ism@b0e767000ec2_codetags/r0
   traces r1: 538 records -> assetgen_meta\traces\assets_heldout26\m7e194es0ism@b0e767000ec2_codetags/r1
   traces r2: 493 records -> assetgen_meta\traces\assets_heldout26\m7e194es0ism@b0e767000ec2_codetags/r2

modules scored: 26; GT entries per scored list: 189 original, 189 corrected (exact)
LAsset initial (no CWE refinement), same modules: P 0.734 R 0.862 emitted 222 (TP 163, FP 59); corrected GT P 0.734 R 0.862

stack              original GT                                     

## Traced arm m7e194es0ist: the winner restructured, every decision cited

**What changed (the user's eight parts).** The winner `m7e194es0ism` is reorganized into: 1 what the model is given, 2 what an asset is (ASSET_DEFINITION.md section 2), 3 purpose and use-case flows (configure, start, operate, report, read out, lock, reset), 4 the flow graph, 5 four questions per value with a mechanism line (a yes without a line is a hypothesis, not reported), 6 roles along the path (stores / sets / computes / exit port are assets; forwards / gates / selects / constrains are influence points, listed separately), 7 exclusions with a reason, 8 checks. Every rule ASSET_DEFINITION.md section 3 marks as a measured contributor is kept verbatim (list in `hand_arms/m7e194es0ist/build_ist.py`). The two worked examples keep their decisions exactly and now show a map excerpt and the citations (`check_examples.py`: occurrences equal a code table, every citation verified).

**Inputs** (`assetgen_meta/traced_inputs.py`, self-tested; no model): (a) the RTL with original line numbers, comments removed; (b) the code-built relationship map WITH occurrence IDs, one record per line (`at` = occurrence IDs), then a FLOW GRAPH computed by code: edge-symmetry check, paths from every input and constant-valued element to stored elements, outputs and sub-unit connections, with the influence points on each path, control-only inputs, elements with no local use.

**Traceability.** Each reported element cites an occurrence ID and an edge; `assetgen_meta/trace_check.py` verifies each citation against the map (status: verified / edge, role unfit / occurrence only / map gap claimed / invalid / no citation) and writes a readable report per module: `assetgen_meta/traces_ist/m7e194es0ist/r<k>/<module>.md`.

**Pre-registered readings** (`hand_arms/read_trace_arm.py`): R1 at least 80% of references verified; R2 yes answers cite real lines; R3 recall guard (map in input, four questions and a separate list each cost recall before: ismr, ismq, v2sec), displacement into influence points / exclusions, port-field FPs, emissions; R4 CITE row (verified references only, evaluation layer); R5 parse failures. Adoption: Step 1b bar AND R1.

Run **Setup** first. 3 runs x 18 modules = 54 calls of gpt-5-mini; inputs are about twice the size of the winner's, estimated $3-6.

In [21]:
# Traced arm ist: load and check (no API call)
HAND_VERSION_T = "m7e194es0ist"
HAND_DIR_T     = Path("assetgen_meta/hand_arms") / HAND_VERSION_T
HAND_PROMPT_T  = (HAND_DIR_T / "exec_prompt.txt").read_text(encoding="utf-8")
_info_t = json.loads((HAND_DIR_T / "build_info.json").read_text(encoding="utf-8"))
_chk_t  = mt.check_exec_prompt(HAND_PROMPT_T, mt.corpus_names(), HAND_VERSION_T)
_MODULES = mt.rtl_modules()
_TIN = Path("assetgen_meta/traced_inputs/tuning")

def _build_traced(stem, rtl):
    """The traced input for this module (numbered RTL + map with occurrence IDs + code flow graph); the comment-stripped
    RTL passed in by run_version is not used."""
    return (_TIN / f"{stem}.txt").read_text(encoding="utf-8")

_have = sum((_TIN / f"{s}.txt").exists() for s, _p in _MODULES)
print(f"{HAND_VERSION_T}: sha {mt.sha12(HAND_PROMPT_T)} (build record {_info_t['system_sha12']}), {len(HAND_PROMPT_T):,} chars; checks:",
      "PASS" if _chk_t["ok"] else _chk_t["problems"], f"| traced inputs {_have}/{len(_MODULES)}")
assert _chk_t["ok"] and mt.sha12(HAND_PROMPT_T) == _info_t["system_sha12"] and _have == len(_MODULES), "do not run"


m7e194es0ist: sha fa07ad720438 (build record fa07ad720438), 193,492 chars; checks: PASS | traced inputs 18/18


In [22]:
# Traced arm ist: 3 runs on the executor (gpt-5-mini, effort high). Cached modules are skipped.
from concurrent.futures import ThreadPoolExecutor, as_completed
_meta_t = {"stage": "hand arm: winner restructured into eight parts, every decision cited (occurrence ID + edge)",
           "build": str(HAND_DIR_T / "build_info.json"), "system_sha12": _info_t["system_sha12"]}
with ThreadPoolExecutor(max_workers=3) as _pool:
    _futs = {_pool.submit(mt.run_version, client, HAND_VERSION_T, _rep, HAND_PROMPT_T, _MODULES, model="gpt-5-mini",
                          extra_meta=_meta_t, workers=6, build=_build_traced,
                          input_note="numbered RTL + relationship map with occurrence IDs + code flow graph (assetgen_meta/traced_inputs)"): _rep
             for _rep in range(3)}
    for _f in as_completed(_futs):
        _s = _f.result()
        print(f"{HAND_VERSION_T} r{_futs[_f]}: {_s['modules_done']}/{len(_MODULES)} modules, tokens {_s['tokens']}, "
              f"validation issues {_s['issues']} {_s['issue_kinds']}")
        for _k, _n in _s["notes"].items():
            if _n not in ("ok", "cached"):
                print("   ", _k, _n)


m7e194es0ist r0: 18/18 modules, tokens {'in': 1544328, 'out': 568778}, validation issues 1 {'bad_realization': 1}
m7e194es0ist r2: 18/18 modules, tokens {'in': 1544328, 'out': 584182}, validation issues 0 {}
m7e194es0ist r1: 18/18 modules, tokens {'in': 1544328, 'out': 591046}, validation issues 1 {'bad_realization': 1}


In [23]:
# Traced arm ist: score against the winner, the pre-registered readings, and the trace reports (no API call)
sys.path.insert(0, "assetgen_meta/hand_arms")
import score_hand_arm as sha, read_trace_arm as rta, trace_check as tc
importlib.reload(sha); importlib.reload(rta); importlib.reload(tc)
_ = sha.score(HAND_VERSION_T, refs=("v2x3r8", "m7e194es0ism"))
rta.read(HAND_VERSION_T, "m7e194es0ism")
print("example report:", sorted(Path("assetgen_meta/traces_ist", HAND_VERSION_T, "r0").glob("neorv32_wdt.md")))


15 modules, 111 ground-truth entries (exact); runs {'v2x3r8': 3, 'm7e194es0ism': 3, 'm7e194es0ist': 3}

1) precision / recall / items per run (strict)
   v2x3r8         P 0.296 R 0.853 items  319.7 (2.88x GT) | per run P [0.289, 0.303, 0.295] R [0.793, 0.901, 0.865] items [304, 330, 325]
   m7e194es0ism   P 0.344 R 0.847 items  273.3 (2.46x GT) | per run P [0.347, 0.332, 0.353] R [0.856, 0.82, 0.865] items [274, 274, 272]
   m7e194es0ist   P 0.359 R 0.691 items  213.7 (1.92x GT) | per run P [0.361, 0.363, 0.353] R [0.676, 0.694, 0.703] items [208, 212, 221]

2) STEP 1b VERDICT: recall 0.691 < 0.83; precision 0.359 >= 0.326 -> does NOT count

4) ground-truth hits and false positives per run, by element kind
   kind                               GT | hits/run       v2x3r8 m7e194es0ism m7e194es0ist | FP/run       v2x3r8 m7e194es0ism m7e194es0ist
   absent from closed set              1 |          0.0          0.0          0.0 |                 2.3          1.0          0.0
   clock/reset 

## Traced arm ist: did the model use the map and answer the four questions? What did it miss?

Run after the traced arm's three runs (the cells above).

**Behaviour table (code, no API call).** The earlier audit found that the current prompt and its map arm answer no question (Integrity on 315 of 318 concepts, 0 negative answers) and cite no map fact (0 of 162). This table measures the same things on every version over the 15 reference modules: objectives, questions answered and negative answers, confidentiality / availability raised, concepts whose reasoning cites a map fact (a relationship type as the map writes it, an occurrence ID or a line number; the plain words "gates" or "copies" do not count), and for the traced arm the references whose occurrence + edge citation verifies. Earlier versions: `m7e194es0ism` (current), `m7e194es0ismr` (map in input), `m7e194es0ismq` (four questions without lines).

**Digests (code).** Per module, exact facts from the traces: each concept with its four answers and line text, each reference with its cited line, edge and citation status; and the blind spots: reference entries missed and where each ended up (influence point, exclusion and its reason, only in a flow path, in a hypothesis, nowhere), flow-graph elements never considered, reported elements the reference does not list, citations that do not verify, run-to-run instability. `assetgen_meta/traces_ist/m7e194es0ist/_digest/`.

**Summaries (LLM, META_MODEL).** One call per module on its digest, then one synthesis. Each summary may use only digest facts and marks judgements as "Interpretation:"; code then lists any element name a summary uses that is not in its digest. `assetgen_meta/traces_ist/m7e194es0ist/_summaries/` (`_overall.md` first). About 16 calls; cached summaries are skipped, so the cell can be re-run.

In [24]:
# Traced arm ist: behaviour against the earlier versions, and the digests (no API call)
import trace_digest as td
importlib.reload(td)
_beh = td.behaviour_table(("m7e194es0ism", "m7e194es0ismr", "m7e194es0ismq", HAND_VERSION_T))
_dig = td.write_digests(HAND_VERSION_T)


behaviour over the 15 reference modules, all runs of each version (counts are totals over the runs)
   version          runs    P     R  emit/run concepts  Integrity  C-yes  A-yes  U-yes  answers  'no'  4x yes  map-fact text  verified cites  hypotheses  exclusions
   m7e194es0ism        3 0.344 0.847    273.3      349    341/349      0      0      0        0     0       0       0/349                 -           0           0
   m7e194es0ismr       3 0.353 0.778    244.7      377    371/377      0      0      0        0     0       0       0/377                 -           0           0
   m7e194es0ismq       3 0.352 0.754    237.7      331    327/331      0    199     76     1324   722       0       0/331                 -           0          22
   m7e194es0ist        3 0.359 0.691    213.7      276    272/276      0    271     39     1104   520       0     276/276           724/771          65         440
digests: 41 modules -> E:\jobs\ff\test\Prepoison_subset\assetgen_meta\traces_is

In [25]:
# Traced arm ist: LLM summaries of the traces (META_MODEL; one call per module + one synthesis; cached files skipped)
_val = td.summarize(client, META_MODEL, HAND_VERSION_T, effort="high")


digests: 41 modules -> E:\jobs\ff\test\Prepoison_subset\assetgen_meta\traces_ist\m7e194es0ist\_digest
   where missed reference entries ended up (run-slots): {'influence point': 30, 'only in a flow path': 14, "mentioned in a concept's text only": 11, 'nowhere in the output': 45, 'excluded': 3}
   flow-graph elements never considered (element-runs): 224
   citation statuses: {'verified': 724, 'occurrence only': 11, 'edge, role unfit': 31, 'map gap claimed': 5}
   summary neorv32_cpu_alu: completed (tokens in 659, out 1272)
   summary neorv32_cpu: completed (tokens in 11397, out 3358)
   summary neorv32_cpu_control: completed (tokens in 680, out 1422)
   summary neorv32_cache: completed (tokens in 17063, out 4742)
   summary neorv32_cpu_cp_bitmanip: completed (tokens in 607, out 980)
   summary neorv32_cpu_counters: completed (tokens in 653, out 1232)
   summary neorv32_bus: completed (tokens in 19003, out 5624)
   summary neorv32_cpu_cp_cond: completed (tokens in 588, out 955)
   summar

In [26]:
# Traced arm ist: read the synthesis and the validation (no API call)
_sd = Path("assetgen_meta/traces_ist", HAND_VERSION_T, "_summaries")
print((_sd / "_overall.md").read_text(encoding="utf-8"))
print("\nnames used by a summary that are not in its digest:",
      {m: v for m, v in json.loads((_sd / "_validation.json").read_text(encoding="utf-8")).items() if v} or "none")
print("per-module summaries:", sorted(p.name for p in _sd.glob("neorv32_*.md")))


# Cross-module synthesis: `m7e194es0ist`

**Evidence scope:** 41 module summaries are supplied. **15 modules have traces for 3 of 3 runs; 26 have no traces for any run**—45 available and 78 missing module-run records. Behavioural findings below concern the 15 observed modules. Missing traces are not counted as executor omissions.

## 1. How the executor reasons now

Across **15 of 15 observed modules**, the summaries report agreement on the module’s broad function in **3 of 3 runs**. They describe flows through configuration, operation, storage, readback, outputs, and reset, as applicable.

- **15/15 modules:** values are grouped into concepts, assessed against the four questions, assigned roles, and supported with relationship-map citations.
- **14/15 modules:** influence points are recorded; `neorv32_sys` is the exception.
- **13/15 modules:** confidentiality hypotheses identify missing system-policy premises; `neorv32_hwspinlock` and `neorv32_sys` are the exceptions.
- The current b

## Arm m7e194es0ist2: generation without the questions, then a separate security-engineer CIA call

**Why (measured).** `ist` reached P 0.359 / R 0.691 (current prompt 0.344 / 0.847). The earlier arm `ismrq` (map + four questions, no influence list) scored the same, 0.360 / 0.697: the recall cost is the map in the input (about -0.07, `ismr`) plus the four questions inside the generation call (about -0.09, `ismq`). The influence list and side lists were where the losses landed (as in `v2sec`); answers doubled in length while concepts fell 21%; bus ports were reported through a "cite a field" route I added (98 of 112 transport references). The confidentiality answers were "no" on 276 of 276 concepts because the inherited criterion asked whether the RTL restricts read-out, which is not the question IEEE P3164 asks.

**Call 1, generation** (`hand_arms/m7e194es0ist2/exec_prompt.txt`): `ist` without the four questions, influence points, hypotheses, exclusions and flow paths; the current prompt's established-value criteria restored (minus the read-out-restriction sentence); citations kept (occurrence + edge, 94% verified in `ist`); transport records can no longer be cited through a field; a signal a sub-unit drives may cite that connection as "computes" (connection directions now come from the sub-unit's declaration: `assetgen_meta/traced_inputs_v2/`). Two fixes for modules that wrap sub-units (a Claude pilot listed 1 of 14 reference entries on cpu): a map-reading correction (signals wired to sub-units are this entity's elements; "not assigned" means no assignment in this file), and IEEE P3164's integrity assumption scoped to values that cross the entity's boundary to a consumer that is not supplied (pilot: cpu 13 of 14). The assumption is recorded in `ASSET_DEFINITION.md` section 6. The worked examples keep their decisions exactly (`examples_ist2.py`, 7 checks).

**Call 2, CIA labelling** (`cia_instructions.md`, `assetgen_meta/cia_label.py`): on the fixed list, one call per run and module answers the four questions the way an SoC security engineer does for an IP of unknown use (IEEE P3164 3.1.1: assume the integrator may need the IP's information kept confidential and its state intact; find writers, observers and blockers in the RTL). Answers are yes-RTL / yes-assumed / no / unknown; every yes and every no cites an RTL line; P3164's worked judgements (including its "no" cases) teach the reasoning. It never changes the list, so it cannot move P or R; it sets the objective and feeds the trace reports.

**Pre-registered readings** (`hand_arms/read_ist2.py`): R0 scores with LAsset spec+RTL and RTL-only rows; R1 format + citations verified >= 90%; R2 R >= 0.817 and P >= 0.344 (adoption: Step 1b bar); R3 concepts per run >= 105; R4 the influence-parked slots come back (>= 12 of 19), emitted <= 285; R5 transport FP slots <= 5, port fields <= 2, cpu_pmp `ctrl_i` 3/3; R6 cpu sub-unit values; R7 map-cost check; R8 lost entries vs the current prompt <= 8; R9 code rows. CIA: C1 no answer above 90% on any question; C2 >= 90% of yes/no answers cite a real line; C3 objective mix next to the reference's; C4 dropping concepts labelled "none" against random thinning.

Run **Setup** first, then the cells below in order. Generation: 54 calls of gpt-5-mini; labelling: 54 calls of gpt-5-mini; summaries: about 16 calls of META_MODEL. Each cell skips finished work, so it can be re-run after an error.

In [28]:
# ist2: load and check both prompts (no API call)
import cia_label as cl
importlib.reload(cl)
HAND_VERSION_T2 = "m7e194es0ist2"
HAND_DIR_T2     = Path("assetgen_meta/hand_arms") / HAND_VERSION_T2
HAND_PROMPT_T2  = (HAND_DIR_T2 / "exec_prompt.txt").read_text(encoding="utf-8")
_info_t2 = json.loads((HAND_DIR_T2 / "build_info.json").read_text(encoding="utf-8"))
_chk_t2  = mt.check_exec_prompt(HAND_PROMPT_T2, mt.corpus_names(), HAND_VERSION_T2)
_cia_t2  = cl.PROMPT.read_text(encoding="utf-8")
_MODULES = mt.rtl_modules()
_TIN2 = Path("assetgen_meta/traced_inputs_v2/tuning")

def _build_traced2(stem, rtl):
    """The v2 traced input (numbered RTL + map with occurrence IDs and resolved connection directions + flow graph)."""
    return (_TIN2 / f"{stem}.txt").read_text(encoding="utf-8")

_have2 = sum((_TIN2 / f"{s}.txt").exists() for s, _p in _MODULES)
print(f"{HAND_VERSION_T2}: generation sha {mt.sha12(HAND_PROMPT_T2)} (build {_info_t2['system_sha12']}), {len(HAND_PROMPT_T2):,} chars, checks:",
      "PASS" if _chk_t2["ok"] else _chk_t2["problems"], f"| CIA prompt sha {mt.sha12(_cia_t2)} (build {_info_t2['cia_sha12']}) | inputs {_have2}/{len(_MODULES)}")
assert _chk_t2["ok"] and mt.sha12(HAND_PROMPT_T2) == _info_t2["system_sha12"] and mt.sha12(_cia_t2) == _info_t2["cia_sha12"] \
    and _have2 == len(_MODULES), "do not run"


m7e194es0ist2: generation sha e5fe4918c22b (build e5fe4918c22b), 178,439 chars, checks: PASS | CIA prompt sha be31fc3e1e9a (build be31fc3e1e9a) | inputs 18/18


In [29]:
# ist2: call 1, generation: 3 runs on the executor (gpt-5-mini, effort high). Cached modules are skipped.
from concurrent.futures import ThreadPoolExecutor, as_completed
_meta_t2 = {"stage": "ist2 generation: no questions or side lists; established-value criteria; transport and sub-unit citation fixes",
            "build": str(HAND_DIR_T2 / "build_info.json"), "system_sha12": _info_t2["system_sha12"]}
with ThreadPoolExecutor(max_workers=3) as _pool:
    _futs = {_pool.submit(mt.run_version, client, HAND_VERSION_T2, _rep, HAND_PROMPT_T2, _MODULES, model="gpt-5-mini",
                          extra_meta=_meta_t2, workers=6, build=_build_traced2,
                          input_note="numbered RTL + relationship map (occurrence IDs, resolved connection directions) + code flow graph (assetgen_meta/traced_inputs_v2)"): _rep
             for _rep in range(3)}
    for _f in as_completed(_futs):
        _s = _f.result()
        print(f"{HAND_VERSION_T2} r{_futs[_f]}: {_s['modules_done']}/{len(_MODULES)} modules, tokens {_s['tokens']}, "
              f"validation issues {_s['issues']} {_s['issue_kinds']}")
        for _k, _n in _s["notes"].items():
            if _n not in ("ok", "cached"):
                print("   ", _k, _n)


m7e194es0ist2 r1: 18/18 modules, tokens {'in': 1448172, 'out': 435285}, validation issues 0 {}
m7e194es0ist2 r0: 18/18 modules, tokens {'in': 1448172, 'out': 479581}, validation issues 0 {}
m7e194es0ist2 r2: 18/18 modules, tokens {'in': 1448172, 'out': 464137}, validation issues 0 {}


In [30]:
# ist2: call 2, CIA labelling: one call per run and module on the fixed list (gpt-5-mini). Labelled modules are skipped.
_lab = cl.label(client, HAND_VERSION_T2, reps=(0, 1, 2), model="gpt-5-mini", workers=6, inputs="traced_inputs_v2")


   r0 neorv32_boot_rom: ok (2 problems)
   r0 neorv32_cpu: ok (1 problems)
   r0 neorv32_package: no concepts
   r0 neorv32_hwspinlock: ok (2 problems)
   r0 neorv32_fifo: ok (1 problems)
   r0 neorv32_spi: ok (2 problems)
   r0 neorv32_wdt: ok (14 problems)
   r1 neorv32_cpu: ok (2 problems)
   r1 neorv32_package: no concepts
   r1 neorv32_cpu_pmp: ok (5 problems)
   r1 neorv32_hwspinlock: ok (2 problems)
   r1 neorv32_fifo: ok (1 problems)
   r1 neorv32_sys: ok (5 problems)
   r2 neorv32_boot_rom: ok (3 problems)
   r2 neorv32_hwspinlock: ok (2 problems)
   r2 neorv32_package: no concepts
   r2 neorv32_imem: ok (4 problems)
   r2 neorv32_twi: ok (1 problems)
labelling m7e194es0ist2: 54 calls, tokens {'in': 1226979, 'out': 1192192}


In [31]:
# ist2: scores against the current prompt and ist, the pre-registered readings, and the trace reports (no API call)
sys.path.insert(0, "assetgen_meta/hand_arms")
import trace_digest as td, cia_label as cl, score_hand_arm as sha, read_ist2 as ri2, trace_check as tc
importlib.reload(td); importlib.reload(cl)   # first: a stale trace_digest hid the CIA labels ('labelled 0/307')
importlib.reload(sha); importlib.reload(ri2); importlib.reload(tc)
_ = sha.score(HAND_VERSION_T2, refs=("m7e194es0ism", "m7e194es0ist"))
ri2.read(HAND_VERSION_T2)
print("example report:", sorted(Path("assetgen_meta/traces_ist", HAND_VERSION_T2, "r0").glob("neorv32_wdt.md")))


15 modules, 111 ground-truth entries (exact); runs {'m7e194es0ism': 3, 'm7e194es0ist': 3, 'm7e194es0ist2': 3}

1) precision / recall / items per run (strict)
   m7e194es0ism   P 0.344 R 0.847 items  273.3 (2.46x GT) | per run P [0.347, 0.332, 0.353] R [0.856, 0.82, 0.865] items [274, 274, 272]
   m7e194es0ist   P 0.359 R 0.691 items  213.7 (1.92x GT) | per run P [0.361, 0.363, 0.353] R [0.676, 0.694, 0.703] items [208, 212, 221]
   m7e194es0ist2  P 0.343 R 0.733 items  237.3 (2.14x GT) | per run P [0.363, 0.339, 0.326] R [0.775, 0.721, 0.703] items [237, 236, 239]

2) STEP 1b VERDICT: recall 0.733 < 0.83; precision 0.343 >= 0.326 -> does NOT count

4) ground-truth hits and false positives per run, by element kind
   kind                               GT | hits/run m7e194es0ism m7e194es0ist m7e194es0ist | FP/run m7e194es0ism m7e194es0ist m7e194es0ist
   absent from closed set              1 |          0.0          0.0          0.0 |                 1.0          0.0          0.0
   clock

In [32]:
# ist2: behaviour against the earlier versions (map use, the four questions), and the digests (no API call)
import trace_digest as td
importlib.reload(td)
_beh2 = td.behaviour_table(("m7e194es0ism", "m7e194es0ismr", "m7e194es0ismq", "m7e194es0ist", HAND_VERSION_T2))
_dig2 = td.write_digests(HAND_VERSION_T2)


behaviour over the 15 reference modules, all runs of each version (counts are totals over the runs)
   version          runs    P     R  emit/run concepts  Integrity  C-yes  A-yes  U-yes  answers  'no'  4x yes  map-fact text  verified cites  hypotheses  exclusions
   m7e194es0ism        3 0.344 0.847    273.3      349    341/349      0      0      0        0     0       0       0/349                 -           0           0
   m7e194es0ismr       3 0.353 0.778    244.7      377    371/377      0      0      0        0     0       0       0/377                 -           0           0
   m7e194es0ismq       3 0.352 0.754    237.7      331    327/331      0    199     76     1324   722       0       0/331                 -           0          22
   m7e194es0ist        3 0.359 0.691    213.7      276    272/276      0    271     39     1104   520       0     276/276           626/771          65         440
   m7e194es0ist2       3 0.343 0.733    237.3      307    303/307    215    217

In [33]:
# ist2: LLM summaries of the traces (META_MODEL; one call per module + one synthesis; cached files skipped), then the synthesis
_val2 = td.summarize(client, META_MODEL, HAND_VERSION_T2, effort="high")
_sd2 = Path("assetgen_meta/traces_ist", HAND_VERSION_T2, "_summaries")
print((_sd2 / "_overall.md").read_text(encoding="utf-8"))


digests: 15 modules -> E:\jobs\ff\test\Prepoison_subset\assetgen_meta\traces_ist\m7e194es0ist2\_digest
   where missed reference entries ended up (run-slots): {"mentioned in a concept's text only": 48, 'nowhere in the output': 43}
   flow-graph elements never considered (element-runs): 403
   citation statuses: {'occurrence only': 104, 'verified': 706, 'edge, role unfit': 63}
   summary neorv32_cpu_cp_cfu: completed (tokens in 13539, out 3719)
   summary neorv32_bus: completed (tokens in 18699, out 4916)
   summary neorv32_cpu: completed (tokens in 13346, out 5239)
   summary neorv32_cache: completed (tokens in 13724, out 6611)
   summary neorv32_cpu_cp_muldiv: completed (tokens in 13702, out 3327)
   summary neorv32_cpu_pmp: completed (tokens in 15427, out 3409)
   summary neorv32_hwspinlock: completed (tokens in 5431, out 2342)
   summary neorv32_debug_dtm: completed (tokens in 17673, out 4844)
   summary neorv32_imem: completed (tokens in 11176, out 3010)
   summary neorv32_sys: com

<!-- [opt-final] -->
## Final version of the prompt-optimization loop (`prompt_opt/v1`), on a stronger generator

**What it is.** The ist2 generation prompt (cells 50-56) plus the three edits the Claude-agent optimization loop kept
(`assetgen_meta/prompt_opt/OPTIMIZATION_LOG.md`, `prompt_opt/v1/change.md`):
- **A.** A stored register is listed only when its own map records show this entity uses its value: it controls another element, carries data into an output, or feeds a sub-unit.
- **B.** A register that only copies another listed element one clock later is not listed again.
- **C.** An input port wired into a sub-unit whose RTL is not supplied "sets" the value it brings in.

Prompt `assetgen_meta/prompt_opt/v1/exec_prompt.txt` (sha `802ee9a90d41`); the same inputs as ist2 (`assetgen_meta/traced_inputs_v2`).

**Generator.** `EXEC_MODEL` (default `gpt-5.4`, a stronger OpenAI model than gpt-5-mini). Changing the model changes two
things at once, so the ist2 prompt is also run on the same model (`BASE_VERSION`). The readings compare the final version
with that paired baseline. Without it they fall back to ist2 on gpt-5-mini and are labelled confounded.

**What is known, on another executor (Claude agents).**
- **Tuning (mean of 3 runs):** precision (P) 0.371, recall (R) 0.934. The ist2 prompt on Claude: 0.335 / 0.887 (2 runs).
- **Held-out (one pre-registered check):** 0.329 / 0.931, against 0.310 / 0.899 for ist2. That precision gain is within noise.
- **ist2 on gpt-5-mini:** 0.343 / 0.733.

**Readings, fixed before the run** (`assetgen_meta/prompt_opt/read_final.py`; a run counts only when complete):
- **F0.** P / R / F1 against the baseline.
- **F1.** Precision gain: "supported" at +0.03 or more, "direction only" between 0 and +0.03, "not supported" at 0 or below.
- **F2.** Recall loss no worse than -0.03.
- **F3.** Edits A and B followed: re-applied as code filters (as shipped), they remove, per run, at most half of what they remove from the baseline's lists.
- **F4.** How many of the 5 cpu interrupt and debug ports each run lists (edit C), for the final version and the baseline.
- **F5.** The share of listed elements whose citation is verified, for both. The trace reports count every citation, so their share differs.

Run **Setup** (cell 1) first. Generation: 54 calls of `EXEC_MODEL` per version (3 runs x 18 modules). On gpt-5-mini, ist2
used about 1.45 M input and 0.44-0.48 M output tokens per run (cell 52); a larger model's output length may differ.
Labelling: 54 more calls (optional). Each cell skips finished work, so it can be re-run after an error.


In [35]:
# [opt-final] Final version (opt v1): choose the generator, load and check the prompts and inputs (one free model lookup, no generation)
import cia_label as cl
importlib.reload(cl)
EXEC_MODEL    = "gpt-5.4"     # the generator: a stronger OpenAI model than gpt-5-mini (one line to change)
_tag = EXEC_MODEL.replace("gpt-", "g").replace(".", "").replace("-", "")
FINAL_VERSION = f"m7e194es0opt1_{_tag}"     # the final prompt on EXEC_MODEL
BASE_VERSION  = f"m7e194es0ist2_{_tag}"     # the ist2 prompt on EXEC_MODEL: the paired baseline
FINAL_DIR     = Path("assetgen_meta/prompt_opt/v1")
FINAL_PROMPT  = (FINAL_DIR / "exec_prompt.txt").read_text(encoding="utf-8")
BASE_PROMPT   = Path("assetgen_meta/hand_arms/m7e194es0ist2/exec_prompt.txt").read_text(encoding="utf-8")
_info_f = json.loads((FINAL_DIR / "build_info.json").read_text(encoding="utf-8"))
_chk_f  = mt.check_exec_prompt(FINAL_PROMPT, mt.corpus_names(), FINAL_VERSION)
_chk_b  = mt.check_exec_prompt(BASE_PROMPT, mt.corpus_names(), BASE_VERSION)
_MODULES_F = mt.rtl_modules()
_TINF = Path("assetgen_meta/traced_inputs_v2/tuning")

def _build_traced_f(stem, rtl):
    """The same input as ist2: numbered RTL + map (occurrence IDs, resolved connection directions) + flow graph."""
    return (_TINF / f"{stem}.txt").read_text(encoding="utf-8")

_have_f = sum((_TINF / f"{s}.txt").exists() for s, _p in _MODULES_F)
print("generator:", client.models.retrieve(EXEC_MODEL).id, "| versions:", FINAL_VERSION, BASE_VERSION)
print(f"final prompt sha {mt.sha12(FINAL_PROMPT)} (build {_info_f['system_sha12']}), {len(FINAL_PROMPT):,} chars, checks:",
      "PASS" if _chk_f["ok"] else _chk_f["problems"], f"| baseline prompt sha {mt.sha12(BASE_PROMPT)}, checks:",
      "PASS" if _chk_b["ok"] else _chk_b["problems"], f"| inputs {_have_f}/{len(_MODULES_F)}")
assert _chk_f["ok"] and _chk_b["ok"] and mt.sha12(FINAL_PROMPT) == _info_f["system_sha12"] == "802ee9a90d41" \
    and mt.sha12(BASE_PROMPT) == "e5fe4918c22b" and _have_f == len(_MODULES_F), "do not run"

def _generate(version, prompt, stage):
    """3 runs of one prompt on EXEC_MODEL (effort high); cached modules are skipped."""
    from concurrent.futures import ThreadPoolExecutor, as_completed
    meta = {"stage": stage, "executor": EXEC_MODEL, "system_sha12": mt.sha12(prompt)}
    with ThreadPoolExecutor(max_workers=3) as pool:
        futs = {pool.submit(mt.run_version, client, version, rep, prompt, _MODULES_F, model=EXEC_MODEL, extra_meta=meta,
                            workers=6, build=_build_traced_f,
                            input_note="numbered RTL + relationship map (occurrence IDs, resolved connection directions) + code flow graph (assetgen_meta/traced_inputs_v2)"): rep
                for rep in range(3)}
        for f in as_completed(futs):
            s = f.result()
            print(f"{version} r{futs[f]}: {s['modules_done']}/{len(_MODULES_F)} modules, tokens {s['tokens']}, "
                  f"validation issues {s['issues']} {s['issue_kinds']}")
            for k, n in s["notes"].items():
                if n not in ("ok", "cached"):
                    print("   ", k, n)


generator: gpt-5.4 | versions: m7e194es0opt1_g54 m7e194es0ist2_g54
final prompt sha 802ee9a90d41 (build 802ee9a90d41), 180,670 chars, checks: PASS | baseline prompt sha e5fe4918c22b, checks: PASS | inputs 18/18


In [36]:
# [opt-final] Final version: generation on EXEC_MODEL, 3 runs (54 calls). Cached modules are skipped.
_generate(FINAL_VERSION, FINAL_PROMPT, "final version of the prompt-optimization loop (prompt_opt/v1): ist2 generation + edits A, B, C")


m7e194es0opt1_g54 r2: 18/18 modules, tokens {'in': 1456542, 'out': 371832}, validation issues 0 {}
m7e194es0opt1_g54 r0: 18/18 modules, tokens {'in': 1456542, 'out': 385623}, validation issues 0 {}
m7e194es0opt1_g54 r1: 18/18 modules, tokens {'in': 1456542, 'out': 411392}, validation issues 0 {}


In [37]:
# [opt-final] Paired baseline: the ist2 prompt on the same EXEC_MODEL, 3 runs (54 calls). Skip it to save cost; the readings then compare with ist2 on gpt-5-mini and say so.
_generate(BASE_VERSION, BASE_PROMPT, "paired baseline: the ist2 generation prompt on the final version's executor")


m7e194es0ist2_g54 r1: 18/18 modules, tokens {'in': 1448172, 'out': 392089}, validation issues 0 {}
m7e194es0ist2_g54 r0: 18/18 modules, tokens {'in': 1448172, 'out': 375089}, validation issues 0 {}
m7e194es0ist2_g54 r2: 17/18 modules, tokens {'in': 1376345, 'out': 377454}, validation issues 0 {}
    neorv32_wdt ERROR RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}


In [ ]:
# [opt-final] Final version: CIA labelling (optional; one call per run and module). It never changes the list, so P and R do not depend on it.
_lab_f = cl.label(client, FINAL_VERSION, reps=(0, 1, 2), model=EXEC_MODEL, workers=6, inputs="traced_inputs_v2")


In [ ]:
# [opt-final] Final version: scores, the readings fixed before the run, and the trace reports (no API call)
for _p in ("assetgen_meta/hand_arms", "assetgen_meta/prompt_opt"):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import trace_digest as td, cia_label as cl, trace_check as tc, fp_diagnosis as fd, score_hand_arm as sha, read_final as rf
for _m in (td, cl, tc, fd, sha, rf):     # reload all: a stale trace_digest in the kernel hid the CIA labels in cell 54
    importlib.reload(_m)
import eval_assets as ea
_paired = len(rf._runs(BASE_VERSION)) == 3
_ = sha.score(FINAL_VERSION, refs=("m7e194es0ist2",) + ((BASE_VERSION,) if _paired else ()))
print()
_rf = rf.read(FINAL_VERSION, base=BASE_VERSION)
_tr = tc.run(FINAL_VERSION, (0, 1, 2), gt=ea.load_refs()["gt"])
print("trace reports:", Path("assetgen_meta/traces_ist", FINAL_VERSION), {k: v["summary"]["verified_share"] for k, v in _tr.items()})


<!-- [opt-final] -->
## Why precision does not improve (1): what the occurrence IDs and the map say about every listed element

For every element it lists, the model cites an **occurrence ID** (one place the element appears in the RTL) and an
**edge** (a record of the relationship map at that place). These two sections use exactly that evidence, and the map
itself, to ask why false positives (FP: listed, but not in the reference) do not go away. TP means listed and in the
reference. No API call in this section.

Each listed element gets a **signature**: what the map says about it, at three levels.
- **L1:** port or signal, record field or whole, stored or combinational.
- **L2:** L1 plus behaviour. Does it update itself, control another element, carry data into an output, take its value from an input, or feed (or get fed by) a sub-unit?
- **L3:** L2 plus the exact set of record types.

The report is `assetgen_meta/fp_diagnosis/<label>.md`; the code is `assetgen_meta/fp_diagnosis.py`, self-tested against hand-read map lines.

| reading | question |
|---|---|
| D1 | Are FPs misreadings? (the citation status of FPs against TPs) |
| D2 | Do FPs look like hits? (the share of FPs whose signature also holds a TP) |
| D3 | Signature-rate selectors over every map element: list a signature when its reference rate is high enough. It is measured two ways. **In-sample** fits the rates on the very elements it scores. **Leave one module out** learns them on the other modules, which is what a rule would carry to a new module. Both read the best point off the curve of the set being scored, so both are optimistic. This is one form of selector; it does not bound methods that use other information. |
| D3b | The same test for a code filter applied to the model's own list |
| D3c | (held-out only) The filter learnt on tuning and applied once to held-out, with its threshold fixed on tuning |
| D4 | Minimal pairs: a TP and an FP with the same signature, from different modules, with the cited RTL lines side by side |
| D5 | FPs inside concepts that contain no reference element at all |
| D6 | Did the model follow edits A and B? (the FPs those edits, as code filters, would still remove) |
| D7 | FP and TP per family of elements, across versions |


In [ ]:
# [opt-final] FP diagnosis on the tuning set (no API call)
for _p in ("assetgen_meta", "assetgen_meta/prompt_opt"):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import fp_diagnosis as fd
importlib.reload(fd)
from IPython.display import Markdown, display
assert fd.selftest(log=lambda *a: None), "fp_diagnosis self-test failed: do not read the reports"
_fv, _bv = globals().get("FINAL_VERSION", "m7e194es0opt1_g54"), globals().get("BASE_VERSION", "m7e194es0ist2_g54")
_exe = globals().get("EXEC_MODEL", "gpt-5.4")
_runs_of = lambda stem, n: [f"{stem}_r{k}" for k in range(n) if fd.complete(f"{stem}_r{k}", "tuning")]   # complete runs only
FD_SETS = {f"{_exe} final":         _runs_of(f"assets_tuning18_{_fv}", 3),
           f"{_exe} ist2":          _runs_of(f"assets_tuning18_{_bv}", 3),
           "gpt-5-mini ist2":       _runs_of("assets_tuning18_m7e194es0ist2", 3),
           "Claude final":          _runs_of("assets_opt_v1", 3),
           "Claude ist2 prompt":    _runs_of("assets_opt_v0", 2)}
FD_SETS = {k: v for k, v in FD_SETS.items() if v}
FD_PRIMARY = next(k for k in (f"{_exe} final", "gpt-5-mini ist2") if k in FD_SETS)
print("run sets:", {k: len(v) for k, v in FD_SETS.items()}, "| primary:", FD_PRIMARY)
_fd = fd.diagnose(FD_SETS[FD_PRIMARY], "tuning_" + FD_PRIMARY.replace(" ", "_").replace(".", "").replace("-", ""), run_sets=FD_SETS)
display(Markdown((fd.OUT / f"{_fd['label']}.md").read_text(encoding="utf-8")))


In [ ]:
# [opt-final] FP diagnosis on held-out (Claude outputs; exploratory: the pre-registered held-out check was read once)
_fdh = fd.diagnose(["assets_opt_heldout_v1_r0"], "heldout_claude_v1", split="heldout",
                   run_sets={"Claude ist2 prompt held-out": ["assets_opt_heldout_v0_r0"], "Claude final held-out": ["assets_opt_heldout_v1_r0"]},
                   train_runs=FD_SETS.get("Claude final", []))
display(Markdown((fd.OUT / "heldout_claude_v1.md").read_text(encoding="utf-8")))


In [ ]:
# [opt-final] FP diagnosis: precision-recall ceilings against the runs (plot; no API call)
import matplotlib.pyplot as plt
_fig, _axes = plt.subplots(1, 2, figsize=(13, 4.8))
for _ax, (_res, _title) in zip(_axes, ((_fd, f"tuning: {FD_PRIMARY}"), (_fdh, "held-out: Claude final (exploratory)"))):
    for _l, _st in ((1, ":"), (2, "--"), (3, "-")):
        _c = _res["D3"]["levels"][_l]["curve_lomo"]
        _ax.plot([x[1] for x in _c], [x[0] for x in _c], _st, color="0.55", label=f"map-only selector, L{_l}, leave one module out")
        _c = _res["D3b"]["levels"][_l]["curve_lomo"]
        _ax.plot([x[1] for x in _c], [x[0] for x in _c], _st, color="tab:blue", label=f"filter on the model's list, L{_l}, leave one module out")
    for _r in _res["runs"]:
        _ax.plot(_r["R"], _r["P"], "o", color="tab:red")
    _ax.plot([], [], "o", color="tab:red", label="the runs")
    _ax.axhline(0.85, color="tab:green", lw=0.8); _ax.axvline(0.85, color="tab:green", lw=0.8)
    _ax.set(xlim=(0, 1.02), ylim=(0, 1.02), xlabel="recall", ylabel="precision", title=_title)
_axes[0].legend(fontsize=7, loc="upper right")
plt.tight_layout(); plt.show()


<!-- [opt-final] -->
## Why precision does not improve (2): the fault reporter, clustering the relationship types of hits and false positives

**Question.** What can a rule stated over relationship types (a prompt edit or a code filter) do against the false
positives? Where hits and FPs carry the same relationship types in the map, such a rule must drop hits with the FPs;
where they differ, it can drop FPs cleanly. The reporter measures how much of each there is, and what the rules found
on one set of modules do on another.

**What the code does** (`assetgen_meta/fault_reporter.py`, self-tested). Every listed element gets a relationship
profile from the map, at two levels:
- **classes:** data out or in, control out, controlled, reset, sub-unit port, clocks or resets others, updates itself (through data or through control); each with what is on the other end (an internal element, an input port, an output port); plus kind, storage, record field and constant driver;
- **types:** the exact record types.

Then:
- **R1.** The share of hits and of FPs that have each token.
- **R2.** Clusters of similar profiles: average linkage on the Jaccard distance, at several cut heights. Reports the share of FPs in clusters that also hold hits.
- **R3.** Can the profile predict hit vs FP? Area under the ROC curve (AUC; 0.5 = chance, 1 = perfect), in-sample and leave one module out.
- **R4.** Per cluster, the best rule that drops its FPs and keeps its hits. It is tested on every listed element and, when given, on held-out runs.

**What the LLM does** (`REPORTER_MODEL`, default `gpt-5.4`; prompts `assetgen_meta/fault_reporter_prompt.md` and `..._synthesis_prompt.md`).
- **Per cluster (at most 8 calls):** what the members share, why the generator listed the FPs, and what differs between hits and FPs in the cited occurrences. It cites module / element / occurrence ID / line. It may give a candidate rule in the token vocabulary.
- **One synthesis call:** a fault taxonomy, and what rules on relationship types achieve and what they do not.

**What the code checks on the LLM** (R5):
- an answer of the wrong shape is not cached and is reported, so a re-run asks again;
- every cited element is in the cluster with that label;
- every occurrence is an integer, the element's own, and its line matches; whether it is the occurrence the generator cited;
- repeated citations are removed, and too few hits or FPs cited is flagged;
- every rule token exists, and every candidate rule is tested on all listed elements (and held-out);
- a "separable" claim (yes / partly / no) is compared with the best clean rule of the code's search and of the LLM's own rule;
- every number in an answer's text must appear somewhere in what that call was given. This is weaker than matching each number to its quantity.

**Outputs.** `assetgen_meta/fault_reporter/<label>/report.md`, `facts.json`, and `ledger.csv`. The ledger traces every
error to the occurrence the generator cited. Answers are cached, so a re-run costs nothing.

The first cell runs the code part only (no API call). The second adds the LLM.


In [ ]:
# [opt-final] Fault reporter, code part only: profiles, clusters, overlap, separability, rules (no API call)
import fault_reporter as fr
importlib.reload(fr)
assert fr.selftest(log=lambda *a: None), "fault_reporter self-test failed: do not read the reports"
REPORTER_MODEL = "gpt-5.4"
FR_JOBS = {FD_PRIMARY: dict(run_sets=FD_SETS, primary=FD_PRIMARY, label="tuning_" + FD_PRIMARY.replace(" ", "_").replace(".", "").replace("-", "")),
           "Claude final": dict(run_sets=FD_SETS, primary="Claude final", label="tuning_Claude_final",
                                transfer_runs=["assets_opt_heldout_v1_r0"])}
FR_JOBS = {k: v for k, v in FR_JOBS.items() if v["primary"] in FD_SETS}
_fr_code = {k: fr.analyse(**v, llm=False) for k, v in FR_JOBS.items()}
for _k, _f in _fr_code.items():
    _t = (fr.OUT / FR_JOBS[_k]["label"] / "report.md").read_text(encoding="utf-8")
    display(Markdown(_t.split("## R6.")[0]))           # code-only: no R5 or synthesis yet


In [ ]:
# [opt-final] Fault reporter with the LLM (REPORTER_MODEL; at most 8 cluster calls + 1 synthesis per report; cached answers are reused)
import matplotlib.pyplot as plt
_fr = {k: fr.analyse(**v, client=client, model=REPORTER_MODEL) for k, v in FR_JOBS.items()}
for _k, _f in _fr.items():
    display(Markdown((fr.OUT / FR_JOBS[_k]["label"] / "report.md").read_text(encoding="utf-8")))
_fig, _axes = plt.subplots(1, len(_fr), figsize=(6.5 * len(_fr), 5.2))
for _ax, _f in zip(_axes if len(_fr) > 1 else [_axes], _fr.values()):
    fr.plot(_f, ax=_ax)
plt.tight_layout(); plt.show()


## Step 1: occurrence profiles, code where the model is weak

The occurrence-profile experiments in `bahavioral_patterns_of_assets/annotation_pack_elements/` measured exactly where the LLM fails. Run 6 compared the model's enclosing-structure chains with a self-tested code finder over 661 entries (trng and cache):

| structure | in source | wrong end | missing or wrong start | extra |
|---|---|---|---|---|
| **branch** | 408 | **106** | **158** | 8 |
| **if** | 408 | 0 | **112** | 2 |
| architecture body | 519 | 0 | 0 | 0 |
| process | 415 | 0 | 0 | 0 |
| case | 163 | 0 | 6 | 0 |
| port clause / association list / generate | 145 | 0 | 0 | 0 |

The model names the **container** perfectly and loses the **branch**: 476 of 661 chains correct overall. Run 5's hand review named the same failure: `context_span_wrong` 104, `context_structure_wrong` 59 of 662. Extraction was already moved to code in run 6, and then 0 of 662 entries sat on a wrong line.

**So this experiment splits the work.** Code does the occurrence inventory and the structure chain (`step1/occurrence_profile.py`, self-tested on 10 hand-read lines of `neorv32_wdt.vhd`). The model does Role and SITE tags, which code cannot write. Every entry's Structure is then **overwritten with the code chain**, and the number of overwrites is reported, so the correction is visible rather than assumed.

The prompt is the run-6 classify prompt, assembled from the same files that produced it (`occurrence_prompts_v2/3_classify.md` + `annotate_rules.md` + `rulebook.json`).

**First a 5-module check**, drawn at random from the 15 test modules with a fixed seed, and capped at `ELEMENT_CAP` elements each, so the cost of the check stays small. Read the table in the last cell before running the rest.

In [8]:
# Step 1: pick the 5 modules, build the code side (no model, no cost)
import random, importlib
sys.path.insert(0, "step1")
import occurrence_profile as OP, occ_llm as OCC
importlib.reload(OP); importlib.reload(OCC)

OCC_SEED     = 20260921      # fixed, so the draw is reproducible
OCC_N        = 5             # modules in this first check
ELEMENT_CAP  = 40            # elements per module in this check; None = every element
OCC_MODEL    = "gpt-5-mini"
OCC_EFFORT   = "low"         # code does the structures; the model only writes Role and SITE tags
OCC_TIMEOUT  = 180           # seconds per call, so a stuck call fails fast instead of hanging

OP.selftest()                                        # 10 assertions read by hand from RTL_data/neorv32_wdt.vhd
OCC.selftest()                                       # ids, batching, and the merge, checked without a model
ALL15 = sorted(p.stem for p in Path("step1/relation_map").glob("*.json"))
OCC_MODULES = sorted(random.Random(OCC_SEED).sample(ALL15, OCC_N))
OCC_SYS, OCC_SHA, OCC_SAME_AS_RUN6 = OCC.classify_system()
OCC_JOBS = {m: OCC.jobs(m, cap=ELEMENT_CAP, seed=OCC_SEED) for m in OCC_MODULES}

print(f"classify prompt {len(OCC_SYS):,} chars, sha {OCC_SHA}; run 6 recorded 62,904 chars sha {OCC.RUN6_CLASSIFY_SHA}"
      f" -> {'identical' if OCC_SAME_AS_RUN6 else 'same assembly and same rulebook, the .md files were edited later the same day'}")
print(f"modules drawn (seed {OCC_SEED}): {OCC_MODULES}")
_in = _calls = 0
for m, js in OCC_JOBS.items():
    n = sum((len(OCC_SYS) + len(OCC.user_message(j))) // 4 for j in js)
    _in += n; _calls += len(js)
    print(f"   {m:24s} batches {len(js):2d}  elements {sum(len(j['elements']) for j in js):3d}  "
          f"occurrences {sum(len(j['chains']) for j in js):4d}  ~{n:7,} input tokens")
print(f"{_calls} calls, ~{_in:,} input tokens -> about ${_in*0.25/1e6 + _calls*6000*2/1e6:.2f} at gpt-5-mini list "
      f"price (output assumed 6k per call)")

selftest PASS: 10 assertions read by hand from RTL_data/neorv32_wdt.vhd
occ_llm selftest PASS | classify prompt 62,945 chars sha a229c20dfd19 != run 6 (6f4f22e256e3) | wdt: 1 batch(es)
classify prompt 62,945 chars, sha a229c20dfd19; run 6 recorded 62,904 chars sha 6f4f22e256e3 -> same assembly and same rulebook, the .md files were edited later the same day
modules drawn (seed 20260921): ['neorv32_bus', 'neorv32_cache', 'neorv32_cpu_cp_cfu', 'neorv32_uart', 'neorv32_wdt']
   neorv32_bus              batches  8  elements  40  occurrences  234  ~231,834 input tokens
   neorv32_cache            batches  5  elements  40  occurrences  193  ~114,885 input tokens
   neorv32_cpu_cp_cfu       batches  3  elements  29  occurrences  148  ~ 68,269 input tokens
   neorv32_uart             batches  7  elements  40  occurrences  332  ~171,211 input tokens
   neorv32_wdt              batches  3  elements  31  occurrences  158  ~ 62,347 input tokens
26 calls, ~648,546 input tokens -> about $0.47 at gpt-

In [9]:
# Step 1: run the model for Role and SITE tags, then correct every Structure with the code chain
RUN_OCC = True               # set True after reading the estimate above
OCC_OUT = Path("step1/occ_llm_out"); OCC_OUT.mkdir(exist_ok=True)
OCC_RESULT, _tok = {}, {"in": 0, "out": 0}
_client = client.with_options(timeout=OCC_TIMEOUT, max_retries=1)   # fail fast; call_text retries on top
if RUN_OCC:
    for m, js in OCC_JOBS.items():
        for j in js:
            f = OCC_OUT / f"{m}__b{j['batch']:02d}.json"
            if f.exists():
                _c = json.loads(f.read_text(encoding="utf-8"))
                if _c.get("stats", {}).get("answered", 0) > 0:
                    OCC_RESULT[f.stem] = _c; print(f"{f.stem}: cached"); continue
                print(f"{f.stem}: cached but nothing joined - running it again")
            print(f"{f.stem}: calling {OCC_MODEL} ({len(j['chains'])} occurrences, "
                  f"~{(len(OCC_SYS)+len(OCC.user_message(j)))//4:,} input tokens) ...", flush=True)
            t0 = time.time()
            try:
                txt, usage, status = mt.call_text(_client, OCC_MODEL, OCC_SYS, OCC.user_message(j),
                                                  OCC_EFFORT, 65536, retries=2, json_mode=True)
            except Exception as e:
                print(f"   FAILED after {round(time.time()-t0)}s: {type(e).__name__}: {str(e)[:160]}"); continue
            _tok["in"] += usage["in"]; _tok["out"] += usage["out"]
            (OCC_OUT / f"{f.stem}.raw.json").write_text(txt, encoding="utf-8")   # keep the answer, so a
            merged = OCC.merge(j, mt.loads(txt))                                #   mismatch never needs a re-run
            merged.update(prompt_sha=OCC_SHA, model=OCC_MODEL, effort=OCC_EFFORT, status=status)
            if status != "completed":
                print(f"   WARNING: status {status} - the answer may be cut off")
            f.write_text(json.dumps(merged, indent=1), encoding="utf-8")
            OCC_RESULT[f.stem] = merged
            s = merged["stats"]
            print(f"   {round(time.time()-t0):3d}s  answered {s['answered']}/{s['occurrences']}  "
                  f"missing {s['missing']}  unknown {s['unknown element']}  added {s['added by the model']}  "
                  f"corrected {s['corrected']} "
                  f"(same {s['same']}, unparseable {s['unparseable']})", flush=True)
    print(f"tokens in {_tok['in']:,} out {_tok['out']:,}  ~${_tok['in']*0.25/1e6 + _tok['out']*2/1e6:.2f}")
else:
    for f in sorted(p for p in OCC_OUT.glob("*.json") if not p.name.endswith(".raw.json")):
        OCC_RESULT[f.stem] = json.loads(f.read_text(encoding="utf-8"))
    print("RUN_OCC is False." + (f" {len(OCC_RESULT)} cached batch(es) loaded." if OCC_RESULT else " Nothing to score yet."))

neorv32_bus__b00: calling gpt-5-mini (50 occurrences, ~30,014 input tokens) ...
   108s  answered 50/50  missing 0  unknown 0  added 0  corrected 32 (same 18, unparseable 0)
neorv32_bus__b01: calling gpt-5-mini (39 occurrences, ~29,430 input tokens) ...
    45s  answered 11/39  missing 28  unknown 0  added 0  corrected 2 (same 9, unparseable 0)
neorv32_bus__b02: calling gpt-5-mini (17 occurrences, ~28,513 input tokens) ...
    68s  answered 17/17  missing 0  unknown 0  added 0  corrected 8 (same 9, unparseable 0)
neorv32_bus__b03: calling gpt-5-mini (33 occurrences, ~29,102 input tokens) ...
    28s  answered 10/33  missing 23  unknown 0  added 0  corrected 5 (same 5, unparseable 0)
neorv32_bus__b04: calling gpt-5-mini (39 occurrences, ~29,468 input tokens) ...
    74s  answered 39/39  missing 0  unknown 0  added 0  corrected 5 (same 34, unparseable 0)
neorv32_bus__b05: calling gpt-5-mini (25 occurrences, ~28,590 input tokens) ...
    18s  answered 2/25  missing 23  unknown 0  added 0 

In [11]:
# Step 1: how did the split do? One row per module, batches summed
if OCC_RESULT:
    KEYS = ("occurrences", "answered", "missing", "unknown element", "added by the model", "corrected", "same", "unparseable",
            "branch corrections")
    per = {}
    for r in OCC_RESULT.values():
        d = per.setdefault(r["module"], {k: 0 for k in KEYS})
        for k in KEYS: d[k] += r["stats"].get(k, 0)
    tot = {k: sum(d[k] for d in per.values()) for k in KEYS}
    print(f"{'module':24s} {'occ':>5s} {'answered':>9s} {'missing':>8s} {'unknown':>8s} {'added':>6s} "
          f"{'corrected':>10s} {'same':>6s} {'unparse':>8s}")
    for m in sorted(per):
        d = per[m]
        print(f"{m:24s} {d['occurrences']:5d} {d['answered']:9d} {d['missing']:8d} {d['unknown element']:8d} {d['added by the model']:6d} "
              f"{d['corrected']:10d} {d['same']:6d} {d['unparseable']:8d}")
    print(f"\n{'TOTAL':24s} {tot['occurrences']:5d} {tot['answered']:9d} {tot['missing']:8d} {tot['unknown element']:8d} {tot['added by the model']:6d} "
          f"{tot['corrected']:10d} {tot['same']:6d} {tot['unparseable']:8d}")
    a = max(tot["answered"], 1)
    print(f"\nstructures the code had to correct: {tot['corrected']}/{a} of the answered ({tot['corrected']/a:.0%})"
          f"   <- run 6 measured 185 of 661 (28%) wrong")
    print(f"of those, occurrences whose branch or if level changed: {tot['branch corrections']}"
          f"   <- the audit says that is where the model loses it")
    print(f"occurrences never answered: {tot['missing']}/{tot['occurrences']} "
          f"({tot['missing']/max(tot['occurrences'],1):.0%}); elements outside the closed set: {tot['unknown element']}; occurrences the model added: {tot['added by the model']}")
    _ex = [e for r in OCC_RESULT.values() for e in r["entries"]
           if e["Structure"] and e["Structure (model, replaced)"] and OCC.triples(e["Structure (model, replaced)"]) != OCC.triples(e["Structure"])]
    for e in _ex[:3]:
        print(f"\n  example correction: {e['Element']} line {e['Line']}\n    model: {str(e['Structure (model, replaced)'])[:150]}"
              f"\n    code : {e['Structure'][:150]}")
else:
    print("no results yet - set RUN_OCC = True in the cell above")

module                     occ  answered  missing  unknown  added  corrected   same  unparse
neorv32_bus                234       160       74        0      0         60    100        0
neorv32_cache              193       191        2        0      0         94     97        0
neorv32_cpu_cp_cfu         148       148        0        0      5         80     68        0
neorv32_uart               332       332        0        0      0        262     70        0
neorv32_wdt                158       158        0        0      0        132     26        0

TOTAL                     1065       989       76        0      5        628    361        0

structures the code had to correct: 628/989 of the answered (63%)   <- run 6 measured 185 of 661 (28%) wrong
of those, occurrences whose branch or if level changed: 543   <- the audit says that is where the model loses it
occurrences never answered: 76/1065 (7%); elements outside the closed set: 0; occurrences the model added: 5

  example corre

### Does `effort` explain the gap?

The first pass ran at `effort="low"` and the model's structure chains matched the code finder's on **37%** of the
answered occurrences; run 6 recorded **72%**. Structures are code-supplied either way, so this does not corrupt the
output, but it is a fair proxy for how carefully the model is doing the part only it can do (Role and SITE tags).

This cell re-runs **one module** at `effort="medium"` into its own folder and prints the two side by side. It costs
about $0.05.

In [ ]:
# Does effort explain the gap? One module at medium effort, compared with the low-effort pass
OCC_MEDIUM_MODULE = "neorv32_wdt"      # the smallest of the five: 158 occurrences in 3 batches
OCC_MEDIUM_DIR = Path("step1/occ_llm_out_medium"); OCC_MEDIUM_DIR.mkdir(exist_ok=True)
RUN_OCC_MEDIUM = True                 # set True to spend about $0.05

_med, _tok2 = {}, {"in": 0, "out": 0}
for j in OCC_MEDIUM_JOBS:
    f = OCC_MEDIUM_DIR / f"{OCC_MEDIUM_MODULE}__b{j['batch']:02d}.json"
    if f.exists() and f.stat().st_size > 0:
        _med[f.stem] = json.loads(f.read_text(encoding="utf-8")); print(f"{f.stem}: cached"); continue
    if not RUN_OCC_MEDIUM:
        continue
    print(f"{f.stem}: calling {OCC_MODEL} at medium effort ({len(j['chains'])} occurrences) ...", flush=True)
    t0 = time.time()
    try:
        txt, usage, status = mt.call_text(_med_client, OCC_MODEL, OCC_SYS, OCC.user_message(j),
                                          "medium", 65536, retries=1, json_mode=True)
    except Exception as e:
        print(f"   FAILED after {round(time.time()-t0)}s: {type(e).__name__}: {str(e)[:160]}"); continue
    _tok2["in"] += usage["in"]; _tok2["out"] += usage["out"]
    (OCC_MEDIUM_DIR / f"{f.stem}.raw.json").write_text(txt, encoding="utf-8")
    m = OCC.merge(j, mt.loads(txt)); m.update(prompt_sha=OCC_SHA, model=OCC_MODEL, effort="medium", status=status)
    f.write_text(json.dumps(m, indent=1), encoding="utf-8"); _med[f.stem] = m
    s = m["stats"]
    print(f"   {round(time.time()-t0):3d}s  answered {s['answered']}/{s['occurrences']}  corrected {s['corrected']} "
          f"(same {s['same']})", flush=True)
if _tok2["in"]:
    print(f"tokens in {_tok2['in']:,} out {_tok2['out']:,}  ~${_tok2['in']*0.25/1e6 + _tok2['out']*2/1e6:.3f}")

def _sum(rs):
    t = {k: 0 for k in ("occurrences", "answered", "missing", "corrected", "same")}
    for r in rs:
        for k in t: t[k] += r["stats"].get(k, 0)
    return t
_low = _sum([r for r in OCC_RESULT.values() if r["module"] == OCC_MEDIUM_MODULE])
_medt = _sum(list(_med.values()))
if _medt["occurrences"]:
    print(f"\n{OCC_MEDIUM_MODULE}: structures matching the code finder")
    for name, t in (("low", _low), ("medium", _medt)):
        a = max(t["answered"], 1)
        print(f"   {name:7s} answered {t['answered']:4d}/{t['occurrences']:4d}  same {t['same']:4d} ({t['same']/a:.0%})  "
              f"corrected {t['corrected']:4d} ({t['corrected']/a:.0%})")
    print("   run 6 recorded 476/661 the same (72%)")
else:
    print("\nset RUN_OCC_MEDIUM = True to run the comparison")

## Occurrence profiles for all 41 ground-truth modules, by code

One file per module in `step1/occurrence_profiles_all/`. Every element of the closed set, every occurrence, with:

| field | what it is | built by |
|---|---|---|
| `line`, `on_line` | the line, and which occurrence on that line | code |
| `roles` | target, read, condition, declaration, port map, clock or reset position | code |
| `structure` | the enclosing chain with spans, e.g. `process 122-137; if 124-136; branch 128-135 (EDGE_CHECK)` | code |
| `statement_id`, `statement_lines`, `statement` | the `;`-terminated statement this occurrence belongs to | code |
| `in_guard`, `guard` | the occurrence is in a control header, and the header text | code |
| `sites` | the rulebook SITE tags | code |

**Why statements are in there.** An outside review (gpt-6-astra) judged the earlier line-only profile *insufficient*
for relation extraction: a line can hold two statements (`dev_26_req_o <= dev_req(26); dev_rsp(26) <= dev_26_rsp_i;`)
and a statement can span lines, so nothing pairs a target with its own operands. `statement_id` closes that gap, and
`guard` separates "X is in the condition governing this assignment" from "X is in the assignment".

**Accuracy, measured, not assumed.** Structures and roles are code and self-tested (10 hand-read assertions; a
20-relation hand check scored 19/20). SITE tags agree with a low-effort LLM on 853 of 989 (86%); where they disagree,
a rulebook adjudication put code right 53% of the time against the model's 38%. The relation-bearing tags are the
strong ones: `IF_COND` 97%, `PROCESS_TRIG` 97%, `WHEN_COND`/`WHEN_EXPR`/`EDGE_CHECK` 100%, `DIRR_ASS` 90%,
`ASSOC_ACTUAL` 90%, `LHS_CONC` 88%, `LHS_PROC` 84%. `INDEX` (19%) and `INDEXED_NAME` (53%) are weak and describe only
how a name is written.

The LLM can still be used as an assistant on top of this (see `step1/occ_llm.py`), but it is no longer required to
produce the profile.

In [ ]:
# Occurrence profiles + SITE tags for all 41 ground-truth modules (code only, no API calls)
import json, sys, importlib
from pathlib import Path
sys.path.insert(0, "step1")
import occurrence_profile as OP, site_tagger as ST
importlib.reload(OP); importlib.reload(ST)
import eval_assets as ea
from collections import Counter

OCC_ALL_DIR = Path("step1/occurrence_profiles_all"); OCC_ALL_DIR.mkdir(exist_ok=True)
OP.selftest()                                   # hand-read assertions run before anything is written

_gt = ea.load_refs()["gt"]
_src = {}
for _d in ("RTL_data", "RTL_heldout"):
    for _p in Path(_d).glob("*.vhd"):
        _src.setdefault(_p.stem, _p)
_mods = sorted(m for m in _gt if m in _src)
print(f"ground-truth modules: {len(_gt)}; RTL found for {len(_mods)}")

_tot = Counter(); _sites = Counter(); _rows = []
for _m in _mods:
    _prof = ST.tag_file(_src[_m])
    (OCC_ALL_DIR / f"{_m}.json").write_text(json.dumps(_prof, indent=1), encoding="utf-8")
    _el = len(_prof["elements"])
    _occ = [o for v in _prof["elements"].values() for o in v["occurrences"]]
    _tagged = sum(1 for o in _occ if o["sites"])
    _stmt = sum(1 for o in _occ if o["statement_id"] is not None)
    _tot.update(elements=_el, occurrences=len(_occ), tagged=_tagged, with_statement=_stmt,
                in_guard=sum(1 for o in _occ if o.get("in_guard")))
    _sites.update(s for o in _occ for s in o["sites"])
    _rows.append((_m, _el, len(_occ), _tagged))
for _m, _el, _oc, _tg in _rows:
    print(f"   {_m:26s} elements {_el:4d}  occurrences {_oc:5d}  with a SITE {_tg:5d} ({_tg/max(_oc,1):3.0%})")
_o = max(_tot["occurrences"], 1)
print(f"\nTOTAL {len(_mods)} modules | elements {_tot['elements']:,} | occurrences {_tot['occurrences']:,}")
print(f"   with a SITE tag      {_tot['tagged']:,} ({_tot['tagged']/_o:.0%})")
print(f"   paired to a statement {_tot['with_statement']:,} ({_tot['with_statement']/_o:.0%})  <- what relation extraction needs")
print(f"   sitting in a guard    {_tot['in_guard']:,} ({_tot['in_guard']/_o:.0%})  <- candidates for GATES / CONTROLLED BY")
print(f"   SITE distribution: {dict(_sites.most_common(12))}")
print(f"\nwritten to {OCC_ALL_DIR}")

## Structure stage — code finds the enclosing structure of every occurrence, BEFORE classify

`knowledge (LLM) → extract (code) → **structure (code)** → classify (LLM) → validate (LLM)`

Run 6 measured the LLM on this job: classify got 185 of 661 chains wrong, all at the if / branch / case-alternative
level, and validate still left 21 wrong after being handed the correct chain for each. So the chain is now computed by
a VHDL grammar (tree-sitter), attached to each extract-inventory row, and shown to classify as read-only context.

Each row gains `construct_chain` (outermost → innermost, each frame `{type, span: [first, last], …}`: e.g.
`{"type": "branch", "kind": "else", "arm": 2, "span": [193, 196]}`, `{"type": "case_alternative", "choice": "S_LOOKUP", …}`),
`path_condition` (what must hold there; an `else`/`elsif` carries `not (…)` of every earlier arm), `context_text`
(the chain as one line, innermost first) and `column`. Comments are separate grammar nodes, so no comment text
can reach a frame or a condition.

Code: `step1/structure_stage.py`. Tested on the three pack modules; the self-test's expected spans were read by hand
from the source, on the exact lines run 6 got wrong. **Named deviation:** LAsset §III-A.3 describes modular RTL
parsing as LLM-based; this deterministic structure profile is an addition to the paper's method.

In [1]:
# Structure stage on the three pack modules: neorv32_boot_rom, neorv32_trng, neorv32_cache (code only, no API calls)
import importlib, sys
sys.path.insert(0, "step1")
import structure_stage as SS
importlib.reload(SS)

jobs_pack, g_pack = SS.extract_inventory(("boot_rom", "trng_cache"))
SS.report(jobs_pack, SS.attach(jobs_pack), g_pack, SS.HAND_PACK,
          examples=[("neorv32_cache", "ctrl_nxt", 200, 0), ("neorv32_cache", "cache_o.we", 194, 0)])
print("\nwritten to step1/structure_stage_out/:", SS.write_outputs(jobs_pack, "step1/structure_stage_out"))

1. parse
   neorv32_boot_rom.vhd       parse errors: none
   neorv32_cache.vhd          parse errors: none
   neorv32_trng.vhd           parse errors: none

2. coverage: extract inventory rows 707 | given a chain 707 | unmatched 0 | matched name differs from 'Name As Written' 0

3. hand-read self-test: 13 chain cases + targeted checks -> PASS

4. against the old regex finder (if / branch / case / alternative / process / loop spans): identical 707 of 707
      0  defect 1: quoted case choice blanked in the old label (old, new)
      0  defect 2: old finder gives a span that ends before it starts
      0  defect 2: lines where occurrences of one element sit in different arms (one chain from a line-based finder, one each from the grammar)
      0  defect 3: grammar says declarative part, old finder says architecture body
     25  defect 4: inside a when-else / with-select, for which the old finder has no frame
            ('neorv32_boot_rom', 'bus_rsp_o', 64, 2)
            ('neorv32_boot

### Structure stage on the three modules where the old regex finder has known defects

The pack modules cannot tell the grammar from the old finder: both are right on them. These three can:

| module | old finder's defect |
|---|---|
| `neorv32_spi` | case choices are quoted literals (`when "101" =>`); the old finder blanks the label to `" "` |
| `neorv32_debug_dtm` | a whole `if … then … else … end if` on one line (123 on); old spans end before they start, and a line-based finder gives both arms' occurrences one chain |
| `neorv32_cpu_cp_crypto` | a function body in the declarative part fools the old `begin` rule; with-select and when-else inside an if-generate get no frame |

Section 3 is a hand-read self-test; section 4 sorts every difference from the old finder into these defects, and lists anything else as unexplained.

In [2]:
# Structure stage on the three modules where the old regex finder has known defects (code only, no API calls)
#   neorv32_spi           case choices are quoted literals ("101"); the old finder blanks them to " "
#   neorv32_debug_dtm     whole if-then-else-end-if written on one line (123-...); old spans end before they start
#   neorv32_cpu_cp_crypto a function body in the declarative part, plus with-select and when-else inside a generate
import importlib, sys
sys.path.insert(0, "step1")
import structure_stage as SS
importlib.reload(SS)

jobs_bug, g_bug = SS.extract_inventory(("neorv32_spi", "neorv32_debug_dtm", "neorv32_cpu_cp_crypto"))
SS.report(jobs_bug, SS.attach(jobs_bug), g_bug, SS.HAND_BUGS,
          examples=[("neorv32_spi", "spi_clk_en", 299, 0),
                    ("neorv32_debug_dtm", "tap_ctrl_state", 123, 0), ("neorv32_debug_dtm", "tap_ctrl_state", 123, 1),
                    ("neorv32_cpu_cp_crypto", "sm4", 268, 0), ("neorv32_cpu_cp_crypto", "rs2_sel", 456, 0)])
print("\nwritten to step1/structure_stage_out/:", SS.write_outputs(jobs_bug, "step1/structure_stage_out"))

1. parse
   neorv32_debug_dtm.vhd      parse errors: none
   neorv32_spi.vhd            parse errors: none
   neorv32_cpu_cp_crypto.vhd  parse errors: none

2. coverage: extract inventory rows 1300 | given a chain 1300 | unmatched 0 | matched name differs from 'Name As Written' 0

3. hand-read self-test: 7 chain cases + targeted checks -> PASS

4. against the old regex finder (if / branch / case / alternative / process / loop spans): identical 1236 of 1300
    118  defect 1: quoted case choice blanked in the old label (old, new)
            ('neorv32_spi', 'spi_dat_i', 310, 2, '\'" "\'', '"110"')
            ('neorv32_spi', 'ctrl', 286, 40, '\'" "\'', '"100"')
            ('neorv32_spi', 'ctrl', 300, 41, '\'" "\'', '"101"')
     64  defect 2: old finder gives a span that ends before it starts
            ('neorv32_debug_dtm', 'tap_sync', 123, 20, [], [('branch', 123, 123)])
            ('neorv32_debug_dtm', 'tap_sync', 124, 21, [], [('branch', 124, 124)])
            ('neorv32_debug_dt

## Record — occurrence profiler v3b (2026-09-29)

**Current version for occurrence profiling.** Notebook `bahavioral_patterns_of_assets/notebooks/occurrence_profiles_v3b.ipynb`, built by `.../occurrence_prompts_v2/rulebook_edits/build_occurrence_notebook_v3b.py`.

### Pipeline

| step | who | what |
|---|---|---|
| 1 knowledge | LLM, once | rulebook `c44db45cd3ed` (21 SITE rules, 16 conflict rules) |
| 2 extract | code | the occurrence inventory: Occurrence ID, line, name as written, line text |
| **2b structure** | **code** | adds **Context** (enclosing constructs, innermost first, with spans) and **Path** (conditions that must hold) to every row, from the tree-sitter VHDL grammar — `step1/structure_stage.py` |
| 3 classify | LLM | Role and SITE tags only; reads Context and Path as fixed facts |
| 4 validate | LLM | re-derives Role and SITE tags; never edits Context or Path |

Model `gpt-5-mini`, effort medium. Batches: at most 20 elements **and at most 85 occurrences**. Prompts `occurrence_prompts_v3/` (classify `94e281c90090`, validate `cb5e024b7d65`); answers `occurrence_profiles_v3b/`.

### What changed, and why

| version | change | reason, measured |
|---|---|---|
| run 6 (v2) | — | classify got 185 of 661 structure chains wrong; validate was handed the correct chain for all 185 and still left 21 wrong. All errors at the if / branch / case-alternative level, all in one oversized batch. |
| v3 | structure owned by code (step 2b); Structure removed from the model's output; prompts revised by gpt-6-astra and checked section by section; 10 rulebook context sentences changed from "Structure" to "Context" (every SITE decision field identical to run 6's `34d10274b852`) | structure is a parsing problem with one right answer |
| v3b | batches capped at 85 occurrences | v3's cache batch 2 held 168 occurrences and lost 11 of `ctrl_nxt`'s 34 rows, 10 more with shifted IDs; every batch of 83 or fewer was clean in run 6 and v3 |

### Results (all re-derived from saved answers; the cell below reproduces the cache rows)

| | run 6 | v3 | v3b |
|---|---|---|---|
| structure chains wrong (validate) | 22 | 0 | 0 |
| cache: right ID and line (validate) | 427 of 427 | 406 of 427 | **427 of 427** |
| cache: `ctrl_nxt` | 34 of 34 | 13 of 34 | **34 of 34** |
| cache: SITE tags identical to run 6 | — | 406 of 406 | **427 of 427** |
| trng+cache: SITE tags identical to run 6 | — | 637 of 641; 4 differ, 2 favour each run (read against the rulebook) | trng not yet run |
| run 6's known wrong tag `debias_data @ 342` | DIRR_ASS | LHS_CONC (fixed) | not yet run (trng) |
| cost | trng+cache $0.7845 (20 calls) | trng+cache $0.3965 (20 calls) | cache $0.2548 (14 calls) |

- **boot_rom:** v3 scored 23 of 23 on the hand key, but every earlier v2 run also scores 23 of 23, so boot_rom does not tell the versions apart.
- **Structure stage on all 44 files:** 0 parse errors; 17,968 of 17,969 extract rows chained (the 1 is an extract bug); 0 lines where a row could get another occurrence's chain; every difference from the old regex finder traced to one of its five defects.

### Open

- **trng under v3b** — `RUN_ON = "trng"`, 8 calls — needed before reporting v3b over all 662 occurrences.
- **Extract counts a name inside a string literal** (1 row in 17,969: `neorv32_xbus` line 136).
- **Mixed conditions** (a signal and a generic in one condition, 907 frames) are not tagged `[static]`.
- **By design, not stated:** whether an occurrence *is* the condition, selector or target. Stating it would largely give classify its SITE.
- **Named deviation:** LAsset §III-A.3 describes modular RTL parsing as LLM-based; the deterministic structure profile is an addition to the paper's method.

In [1]:
# Occurrence profiler v3b: re-derive the recorded numbers from the saved answers (no API call, nothing written)
import json, re
from pathlib import Path

_P = Path("bahavioral_patterns_of_assets/annotation_pack_elements")
_RUNS = {"run 6 (v2)": _P / "occurrence_profiles_v2/validate/f4082229020d",
         "v3":         _P / "occurrence_profiles_v3/validate/cb5e024b7d65",
         "v3b":        _P / "occurrence_profiles_v3b/validate/cb5e024b7d65"}


def _answers(d, module):
    out = {}
    for f in Path(d).glob(f"{module}__*.json"):
        rec = json.loads(f.read_text(encoding="utf-8"))
        for n, rows in json.loads(re.sub(r"^```[a-z]*\n|\n```$", "", rec["raw"].strip())).items():
            out.setdefault((rec["entity"], n), {}).update(
                {(r.get("Occurrence ID"), r.get("Occurrence Lines")): frozenset(r.get("SITE Tagged") or [])
                 for r in rows if isinstance(r, dict)})
    return out


_inv = {}
for f in (_P / "occurrence_profiles_v3b/extract").rglob("neorv32_cache__*.json"):
    rec = json.loads(f.read_text(encoding="utf-8"))
    for n, rows in json.loads(rec["raw"]).items():
        _inv[(rec["entity"], n)] = [(r["Occurrence ID"], r["Occurrence Lines"]) for r in rows]
_A = {k: _answers(d, "neorv32_cache") for k, d in _RUNS.items()}
_N = sum(len(v) for v in _inv.values())
print(f"neorv32_cache, validate step: {_N} inventory occurrences")
print(f"   {'run':12s} {'right ID and line':>18s} {'ctrl_nxt':>9s} {'SITE tags = run 6':>18s}")
for k, A in _A.items():
    hit = [(e, o) for e, os_ in _inv.items() for o in os_ if o in A.get(e, {})]
    cn = sum(1 for o in _inv[("neorv32_cache", "ctrl_nxt")] if o in A.get(("neorv32_cache", "ctrl_nxt"), {}))
    base = _A["run 6 (v2)"]
    both = [(e, o) for e, o in hit if o in base.get(e, {})]
    same = sum(1 for e, o in both if A[e][o] == base[e][o])
    print(f"   {k:12s} {len(hit):>10d} of {_N} {cn:>5d} of 34 {same:>10d} of {len(both)}")

neorv32_cache, validate step: 427 inventory occurrences
   run           right ID and line  ctrl_nxt  SITE tags = run 6
   run 6 (v2)          427 of 427    34 of 34        427 of 427
   v3                  406 of 427    13 of 34        406 of 406
   v3b                 427 of 427    34 of 34        427 of 427


## Relation stage v1: the relationship annotator

An LLM writes the relationships between the elements of an entity, from the occurrence profile.

**Pipeline:** knowledge (LLM) → extract (code) → structure (code) → classify, validate (LLM: SITE, Role) → **relation (LLM)** → merge (code).

**Inputs per call** (prefix first, so the batches of one entity share the cached part):
- the entity's source, comments removed, with line numbers
- the closed set: ports (with mode) and signals
- the occurrence profile of every element of the entity (Context and Path by code, SITE and Role by validate)
- the batch: at most 20 elements and 85 occurrences

**Prompt:** `relation_prompts_v1/relation_system_prompt.md`, written by gpt-6-astra from `META_PROMPT.md` (sha `fa714836cf69`: astra output `520056ab5efd` plus recorded post-edits, see `astra_meta_run.json`). Executor: gpt-5-mini, effort medium.

**The model writes** `name`, `functionality` and `relationship` (`type`, `targets`, `at`, `bits`). It writes both sides of every pair, each at its own occurrence.

**The program writes** `entity`, `kind`, `handling`, `guard`, `partner_at`, `boundary`, `storage`, `constant_drivers`, `configuration`, `connections` and `occurrences`.

**Traceability:** every Occurrence ID in the file (`at`, `partner_at`, `constant_drivers`, `connections`, `configuration`) resolves through the element's `occurrences` list to its source line and text. `partner_at` gives, for each target, the target's occurrences that hold the partner record. `entity` is needed because one file can hold several entities, each with its own Occurrence IDs.

**Final file:** `step1/relation_map_v1/<prompt sha>/<module>.json`
```
{"ports":   [{"name", "entity", "functionality", "handling", "boundary", "storage", "constant_drivers",
              "configuration": [{"condition", "at"}], "connections",
              "relationship": [{"type": "<EDGE>", "targets": [...], "at": [...],
                                "guard": "<GATES/SELECTS/CONSTRAINS and partners>",
                                "partner_at": {"<target>": [...]}, "bits": "<optional>"}],
              "occurrences": [{"id", "line", "text"}]}],
 "signals": [{"name", "entity", "kind", "functionality", "handling", "storage", "constant_drivers",
              "configuration", "connections", "relationship": [...], "occurrences": [...]}]}
```
`handling` appears only on record fields (dotted names).

**Checks after each run** (code):
- **Shape:** each batch element appears once, in the right array.
- **G1:** every `at` is an Occurrence ID of the element, and every target is in the closed set.
- **G3:** each type is allowed for the SITE of its occurrence.
- **Banned words:** none in `functionality`.
- **Partners:** every record has its partner record.
- **G2/G7:** every occurrence × target is linked by rules L1–L5.
- **Agreement:** the model's element pairs, and its typed records (element, occurrence, type, target), are compared with the same table run by code, as recall and precision against that reference. This is agreement, not accuracy.
- **Traceability (stub self-test):** every Occurrence ID in the final file resolves to a line, and every partner is found.

Answers are cached under `step1/relation_out/<prompt sha>/`. A re-run skips a batch whose input has not changed.


In [3]:
# Relation stage v1: audit the prompt, test the plumbing with a stub answer, print the call plan (no API calls)
import importlib, sys
sys.path.insert(0, "step1")
import relation_stage as RS
importlib.reload(RS)

REL_MODULES = ["neorv32_boot_rom", "neorv32_trng", "neorv32_cache"]
assert RS.audit_prompt(), "the prompt fails the project constraints: fix it before calling the model"
for m in REL_MODULES:
    print(f"\n{m}  (stub: a perfect answer built from the code reference)")
    assert RS.selftest_stub(m), f"plumbing broken on {m}"
print("\ncall plan:")
n_calls = sum(RS.run(m, dry=True)["cost"].get("calls", 0) for m in REL_MODULES)
print(f"calls to make: {n_calls}")


   prompt relation_system_prompt.md: 27,214 chars, sha a44eebd83bf7; audit PASS

neorv32_boot_rom  (stub: a perfect answer built from the code reference)
   relationship records (one per occurrence x target) 18: partner written 18, partner missing 0
   element pairs, model vs the table run by code: reference 9, model 9, both 9. Recall 100.0%, precision 100.0%  (agreement with the code reference, exact denominators; not accuracy against a hand label)
   typed records (element, occurrence, type, target), model vs the same reference: reference 18, model 18, both 18. Recall 100.0%, precision 100.0%  (the reference's CONSTRAINS / GATES-operand typing is approximate: read the differences before calling one a model error)
   check failures: none
   wrote step1\relation_out\stub_map\neorv32_boot_rom.json: 19 ports, 2 signals; guarded records with no guard found: 0
   final-format check: entries with wrong fields 0, records with wrong keys 0, Occurrence IDs that do not resolve to a line 0 []
  

In [4]:
# Relation stage v1: annotate (gpt-5-mini), check, merge the program's fields, write the final file
# RUN_ON: "none" makes no call; a module name runs that module; "all" runs the three.
# Start with neorv32_boot_rom (2 calls): read its checks before running the others.
RUN_ON = "neorv32_boot_rom"

from dotenv import load_dotenv
from openai import OpenAI
load_dotenv("API.env")
rel_client = OpenAI().with_options(timeout=900, max_retries=0)

todo = REL_MODULES if RUN_ON == "all" else [m for m in REL_MODULES if m == RUN_ON]
rel_cost, rel_res = {}, {}
for m in todo:
    print(f"\n=== {m}")
    rel_cost[m] = RS.run(m, client=rel_client)["cost"]
    rel_res[m] = RS.check(m)
    RS.merge(m, rel_res[m])
for m, c in rel_cost.items():
    print(f"{m}: {c.get('calls', 0)} calls, tokens in {c.get('in', 0):,} out {c.get('out', 0):,}, "
          f"cost at most ${c.get('usd_upper', 0):.4f} (upper bound: cached input billed at the full rate)")
if not todo:
    print("RUN_ON is 'none': nothing called")



=== neorv32_boot_rom
   calling  neorv32_boot_rom b0: 20 elements, 42 occurrences ...
            done in 47 s, status completed, out 7,645 tokens
   calling  neorv32_boot_rom b1: 1 elements, 3 occurrences ...
            done in 13 s, status completed, out 2,142 tokens
   relationship records (one per occurrence x target) 17: partner written 16, partner missing 1
   element pairs, model vs the table run by code: reference 9, model 9, both 9. Recall 100.0%, precision 100.0%  (agreement with the code reference, exact denominators; not accuracy against a hand label)
   typed records (element, occurrence, type, target), model vs the same reference: reference 18, model 17, both 17. Recall 94.4%, precision 100.0%  (the reference's CONSTRAINS / GATES-operand typing is approximate: read the differences before calling one a model error)
   check failures: none
   diagnostic, reference record the model did not write: 1 of 17 records; e.g. [('neorv32_boot_rom', 'rdata', 2, 'CLOCKED_BY', 'clk_i'